# FedSVLM on FineFake — data preparation and experiments

Notebook 2 of 2 for the term paper **"Federated Learning with Fine-Tuned Small Vision-Language Models for Multimodal Fake News Detection"** (NLP course, Quy Nhon University).

**What this notebook does**
1. Downloads FineFake (Zhou et al., 2024) from the authors' official link and preprocesses it into memory-mapped `.npy` arrays.
2. Trains Light-MFND (CLIP ViT-B/32 + LoRA + cross-attention fusion) centrally, with FedAvg (**FedSVLM**), and as a late-fusion variant (**FedSVLM-LF**), with 4 clients = 4 topics (Politics, Business, Health, Society); Entertainment and Conflict are held out as out-of-domain (OOD) test sets.
3. Analyses a source (platform) shortcut, tests the knowledge embeddings shipped with FineFake, and evaluates source-balanced client resampling.

**Environment.** Google Colab, NVIDIA T4 GPU. Cells F1a–F2 only need a CPU. Shared files live in Google Drive under `MyDrive/NLP_project/fed/` (`init.pt`, the common LoRA initialisation, is created by notebook 1). Every result is appended to `fed/finefake/results_finefake.json` (also in `results/` of this repository).

**Run order.** Top to bottom. Each experiment has its own cell and writes its own checkpoint, so no result overwrites another. Training cells resume from their checkpoint if a session is interrupted.

**Notes on the saved outputs.**
- The experiments were run on 18–20 Nov 2026. Colab sessions were restarted several times (GPU quota), so cells were not all executed in one session; execution counters are therefore cleared.
- Cells K1 and K2 show the outputs of their run on 20 Nov 2026 (the code is unchanged; a later re-execution failed only because the raw `.pkl` was no longer in the session).
- Cell B4a shows its training log; it was later re-executed only to reload the finished checkpoint. Cell B4b was interrupted after round 8 and resumed from the round-5 checkpoint; both logs are kept.

**Glossary of Vietnamese labels that appear in code comments and printed outputs**

| Vietnamese | English |
|---|---|
| Tập trung | Centralized |
| chỉ ảnh / chỉ chữ | image-only / text-only |
| kết hợp muộn | late fusion |
| chủ đề | topic (client) |
| nguồn tin | news source (platform) |
| cân bằng theo nguồn | source-balanced resampling |
| thường | standard (no resampling) |
| AUC gộp | pooled AUC (test + OOD sets) |
| TB trong từng nguồn | mean within-source AUC |
| tri thức | knowledge (embeddings) |
| bỏ trùng | after removing texts duplicated in train |
| vòng | round |

## 0. Setup
Libraries, model definition (Light-MFND and single-modality variants), training and evaluation helpers, shared LoRA initialisation.

In [ ]:
# Ô F0a — Cài thư viện (giống ô 1 của notebook MiRAGeNews)
!pip install -q peft datasets scikit-learn
!pip install -q -U torchao

In [ ]:
# Ô F0b — Mô hình và hàm dùng chung (lấy nguyên từ ô 2, 3, 4, 24, 32 của notebook MiRAGeNews)
import os, copy, time, json, gc, numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
from transformers import CLIPModel
from peft import LoraConfig, get_peft_model
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from google.colab import drive
if not os.path.exists("/content/drive/MyDrive"): drive.mount("/content/drive")
MODEL_NAME = "openai/clip-vit-base-patch32"
device = "cuda" if torch.cuda.is_available() else "cpu"; torch.backends.cuda.matmul.allow_tf32 = True
FED_MIRAGE = "/content/drive/MyDrive/NLP_project/fed"          # để dùng lại init.pt (cùng khởi tạo với MiRAGeNews)

class MM(torch.utils.data.Dataset):                            # đọc dữ liệu memmap (từ ô 2)
    def __init__(s, sp):
        s.pix = np.load(f"{NPY}/{sp}_pix.npy", mmap_mode="r")
        s.ids = np.load(f"{NPY}/{sp}_ids.npy"); s.msk = np.load(f"{NPY}/{sp}_msk.npy"); s.labels = np.load(f"{NPY}/{sp}_lab.npy")
    def __len__(s): return len(s.labels)
    def __getitem__(s, i):
        return dict(input_ids=torch.from_numpy(s.ids[i]), attention_mask=torch.from_numpy(s.msk[i]),
                    pixel_values=torch.from_numpy(np.array(s.pix[i])), labels=torch.tensor(s.labels[i]))

class CLIPForMFND(nn.Module):                                  # Light-MFND (từ ô 3)
    def __init__(self, model_name=MODEL_NAME, num_labels=2, lambda_weight=0.1, delta=0.5):
        super().__init__()
        base = CLIPModel.from_pretrained(model_name); d = base.config.projection_dim
        self.clip = get_peft_model(base, LoraConfig(r=8, lora_alpha=16, target_modules=["q_proj", "v_proj"],
                                                    lora_dropout=0.1, bias="none"))
        self.cross_attention = nn.MultiheadAttention(d, 8, batch_first=True)
        self.W_g = nn.Linear(2 * d, d); self.classifier = nn.Linear(d, num_labels)
        self.lw, self.delta = lambda_weight, delta
    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        o = self.clip(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values, return_dict=True)
        hT, hI = o.text_embeds, o.image_embeds
        z, _ = self.cross_attention(hT.unsqueeze(1), hI.unsqueeze(1), hI.unsqueeze(1)); z = z.squeeze(1)
        hf = torch.sigmoid(self.W_g(torch.cat([hT, hI], 1))) * z
        logits = self.classifier(hf); loss = None
        if labels is not None:
            sim = F.cosine_similarity(hT, hI)
            la = torch.where(labels == 0, 1 - sim, torch.clamp(sim - self.delta, min=0.0)).mean()
            loss = F.cross_entropy(logits, labels) + self.lw * la
        return loss, logits

class CLIPModalMFND(CLIPForMFND):                              # mô hình một nhánh (từ ô 24)
    def __init__(self, mode):
        super().__init__(); self.mode = mode
    def forward(self, input_ids, attention_mask, pixel_values, labels=None):
        o = self.clip(input_ids=input_ids, attention_mask=attention_mask, pixel_values=pixel_values, return_dict=True)
        h = o.image_embeds if self.mode == "image" else o.text_embeds
        logits = self.classifier(h)
        return (F.cross_entropy(logits, labels) if labels is not None else None), logits

def trainable_keys(m): return [n for n, p in m.named_parameters() if p.requires_grad]
def get_state(m, keys):
    P = dict(m.named_parameters()); return {k: P[k].detach().cpu().clone() for k in keys}
def set_state(m, st):
    P = dict(m.named_parameters())
    with torch.no_grad():
        for k, v in st.items(): P[k].copy_(v.to(P[k].device))
def loader(dset, bs, shuffle, idx=None):
    d = Subset(dset, idx) if idx is not None else dset
    return DataLoader(d, batch_size=bs, shuffle=shuffle, num_workers=2, pin_memory=True)
def batch_to(b):
    d = {k: b[k].to(device, non_blocking=True) for k in ("input_ids", "attention_mask", "pixel_values", "labels")}
    d["pixel_values"] = d["pixel_values"].float(); return d

def train_epochs(m, dset, idx=None, epochs=1, lr=1e-5, wd=0.01, bs=16, ref=None, mu=0.0):
    m.train(); named = [(n, p) for n, p in m.named_parameters() if p.requires_grad]; ps = [p for _, p in named]
    refs = [ref[n].to(device) for n, _ in named] if (ref is not None and mu > 0) else None
    opt = torch.optim.AdamW(ps, lr=lr, weight_decay=wd); sc = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    tot, n = 0.0, 0
    for _ in range(epochs):
        for b in loader(dset, bs, True, idx):
            b = batch_to(b)
            with torch.autocast(device, dtype=torch.float16, enabled=(device == "cuda")):
                loss, _ = m(b["input_ids"], b["attention_mask"], b["pixel_values"], b["labels"])
            if refs is not None:
                loss = loss + (mu / 2) * sum(((p - r_) ** 2).sum() for p, r_ in zip(ps, refs))
            opt.zero_grad(set_to_none=True); sc.scale(loss).backward(); sc.step(opt); sc.update()
            tot += loss.item() * len(b["labels"]); n += len(b["labels"])
    return tot / max(n, 1)

@torch.no_grad()
def evaluate(m, dset, bs=64):
    m.eval(); P, Y = [], []
    for b in loader(dset, bs, False):
        b = batch_to(b)
        with torch.autocast("cuda", dtype=torch.float16):
            _, lg = m(b["input_ids"], b["attention_mask"], b["pixel_values"])
        P.append(torch.softmax(lg.float(), 1)[:, 1].cpu()); Y.append(b["labels"].cpu())
    p, y = torch.cat(P).numpy(), torch.cat(Y).numpy(); pred = (p > 0.5).astype(int)
    return dict(acc=accuracy_score(y, pred) * 100, f1=f1_score(y, pred) * 100, auc=roc_auc_score(y, p) * 100)

def save_result(key, val):                                     # (từ ô 4) ghi vào RES_PATH, sẽ đặt ở ô F3
    R = json.load(open(RES_PATH)) if os.path.exists(RES_PATH) else {}
    R[key] = val; json.dump(R, open(RES_PATH, "w"), indent=1); return R

def run_centralized(model, tag, train_fn, seed, save_name, epochs=5, patience=2):   # (từ ô 24)
    set_state(model, INIT); torch.manual_seed(1000 * seed)
    keys = [n for n, p in model.named_parameters() if p.requires_grad]
    best, bad, best_state, t0 = -1.0, 0, None, time.time()
    for ep in range(1, epochs + 1):
        l = train_fn(model, enc["train"], epochs=1); v = evaluate(model, enc["validation"])
        print(f"epoch {ep} loss {l:.4f} | val F1 {v['f1']:.2f} AUC {v['auc']:.2f} | {time.time()-t0:.0f}s")
        if v["f1"] > best: best, bad, best_state = v["f1"], 0, get_state(model, keys)
        else:
            bad += 1
            if bad >= patience: break
    set_state(model, best_state); res = eval_all(model, tag)
    res.update(time_s=time.time() - t0, epochs_run=ep)
    save_result(tag, res); torch.save(best_state, f"{FED}/{save_name}.pt"); return res

def _load_ck(CK, init_g):                                      # (từ ô 32)
    if os.path.exists(CK):
        S = torch.load(CK, weights_only=False); print(f"Tiếp tục từ vòng {S['r']+1}")
        return S["g"], float(S["best"]), S["best_g"], S["LOG"], S["r"]
    return init_g, -1.0, None, [], 0

def _fedavg_loop(model, keys, parts, g, best, best_g, LOG, r0, ROUNDS, seed, CK, mu=0.0):
    torch.manual_seed(1000 * seed + r0); N = sum(len(p) for p in parts)
    for r in range(r0 + 1, ROUNDS + 1):
        t0 = time.time(); acc = {k: torch.zeros_like(v, dtype=torch.float32) for k, v in g.items()}; losses = []
        for p in parts:
            set_state(model, g)
            losses.append(train_epochs(model, enc["train"], p, epochs=1, ref=g, mu=mu))
            st = get_state(model, keys)
            for key in acc: acc[key] += st[key] * (len(p) / N)
        g = acc; set_state(model, g); v = evaluate(model, enc["validation"])
        LOG.append(dict(round=r, val_f1=float(v["f1"]), val_acc=float(v["acc"]), val_auc=float(v["auc"]),
                        loss=float(np.mean(losses)), sec=float(time.time() - t0)))
        print(f"round {r:2d} | loss {np.mean(losses):.4f} | val Acc {v['acc']:.2f} F1 {v['f1']:.2f} AUC {v['auc']:.2f} | {time.time()-t0:.0f}s")
        if v["auc"] > best: best, best_g = float(v["auc"]), copy.deepcopy(g)
        torch.save(dict(g=g, best=float(best), best_g=best_g, LOG=LOG, r=int(r)), CK)
    return best_g, LOG

def _finish(model, best_g, LOG, TAG, mb, path):
    set_state(model, best_g); res = eval_all(model, TAG)
    res.update(rounds_run=len(LOG), best_round=int(np.argmax([x["val_auc"] for x in LOG])) + 1,
               time_s=float(sum(x["sec"] for x in LOG)), update_mb=mb, log=LOG)
    save_result(TAG, res); torch.save(best_g, path); return path

m = CLIPForMFND().to(device); KEYS = trainable_keys(m)
INIT = torch.load(f"{FED_MIRAGE}/init.pt"); set_state(m, INIT)
tr = sum(p.numel() for p in m.parameters() if p.requires_grad); tot = sum(p.numel() for p in m.parameters())
print(f"Trainable {tr:,} / {tot:,} — kỳ vọng 2,067,970 / 153,345,283 | dùng lại init.pt của MiRAGeNews")

Mounted at /content/drive


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  605MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  605MB            

model.safetensors: downloading bytes:           |  0.00B            

Trainable 2,067,970 / 153,345,283 — kỳ vọng 2,067,970 / 153,345,283 | dùng lại init.pt của MiRAGeNews


## 1. Data: download and preprocessing (CPU)
Official FineFake release → label check → topic-based split (6:2:2 in-domain; Entertainment and Conflict as OOD) → CLIP tokenisation and image arrays.

In [ ]:
# Ô F1a — Tải FineFake từ link Google Drive chính thức của tác giả và giải nén (khoảng 3–5 phút)
!pip install -q gdown
import os, glob, zipfile, collections, gdown
ROOT = "/content/finefake"; os.makedirs(ROOT, exist_ok=True)
out = gdown.download(id="16D9ix7ZOisa4VVBznBTBcv1N7TA-jodH", output=f"{ROOT}/", quiet=False)
print("Đã tải:", out, f"{os.path.getsize(out)/1e9:.2f} GB")
zipfile.ZipFile(out).extractall(ROOT); print("Đã giải nén")
ext = collections.Counter(os.path.splitext(f)[1].lower() for f in glob.glob(f"{ROOT}/**/*", recursive=True) if os.path.isfile(f))
print("Số file theo đuôi:", ext.most_common(10))

Downloading...
From (original): https://drive.google.com/uc?id=16D9ix7ZOisa4VVBznBTBcv1N7TA-jodH
From (redirected): https://drive.google.com/uc?id=16D9ix7ZOisa4VVBznBTBcv1N7TA-jodH&confirm=t&uuid=6f29154d-6aad-4a71-bac2-338c8bd4213e
To: /content/finefake/FineFake.zip
100%|██████████| 5.06G/5.06G [01:30<00:00, 55.9MB/s]


Đã tải: /content/finefake/FineFake.zip 5.06 GB
Đã giải nén
Số file theo đuôi: [('.jpeg', 36271), ('.png', 4049), ('.ico', 276), ('.gif', 47), ('.pkl', 1), ('.zip', 1)]


In [ ]:
# Ô F1b — Đọc FineFake.pkl và xem cấu trúc
import os, glob, numpy as np, pandas as pd
ROOT = "/content/finefake"
df = pd.read_pickle(f"{ROOT}/FineFake.pkl")
print("numpy", np.__version__, "| pandas", pd.__version__)
print("Kích thước:", df.shape); print("Cột:", list(df.columns))
print("Ví dụ image_path:", df["image_path"].iloc[:3].tolist())
print("\nNhãn:", df["label"].value_counts().to_dict())
print("Chủ đề:", df["topic"].value_counts().to_dict())
print("Nền tảng:", df["platform"].value_counts().to_dict())
print("Tỉ lệ giả (label=0) theo nền tảng (%):", df.groupby("platform")["label"].apply(lambda s: round((s == 0).mean() * 100, 1)).to_dict())
print("Tỉ lệ giả (label=0) theo chủ đề (%):", df.groupby("topic")["label"].apply(lambda s: round((s == 0).mean() * 100, 1)).to_dict())
ok = sum(os.path.exists(os.path.join(ROOT, str(p))) for p in df["image_path"].head(200))
print(f"Tìm thấy {ok}/200 ảnh khi ghép đường dẫn với {ROOT}")
print("Số mẫu không có image_path:", df["image_path"].isna().sum())

numpy 2.4.6 | pandas 2.2.3
Kích thước: (16909, 13)
Cột: ['text', 'image_path', 'entity_id', 'topic', 'fine-grained label', 'label', 'knowledge_embedding', 'description', 'relation', 'platform', 'author', 'date', 'comment']
Ví dụ image_path: ['Image/cdc_gov/0.ico', 'Image/cdc_gov/1.ico', 'Image/cdc_gov/6.ico']

Nhãn: {0: 9407, 1: 7502}
Chủ đề: {'Politics': 5727, 'Society': 3939, 'Entertainment': 3699, 'Conflict': 1718, 'Business': 1003, 'Health': 710, 'Uncategorized': 113}
Nền tảng: {'snope': 7556, 'reddit': 4048, 'cnn': 2310, 'washingtonpost': 1041, 'twitter': 952, 'apnews': 734, 'cdc_gov': 268}
Tỉ lệ giả (label=0) theo nền tảng (%): {'apnews': 9.3, 'cdc_gov': 1.1, 'cnn': 10.3, 'reddit': 76.3, 'snope': 67.3, 'twitter': 62.1, 'washingtonpost': 32.3}
Tỉ lệ giả (label=0) theo chủ đề (%): {'Business': 52.1, 'Conflict': 49.7, 'Entertainment': 58.9, 'Health': 42.8, 'Politics': 51.9, 'Society': 63.7, 'Uncategorized': 60.2}
Tìm thấy 200/200 ảnh khi ghép đường dẫn với /content/finefake
Số mẫu k

In [ ]:
# Ô F1c — Kiểm tra nhãn nhị phân và nhãn chi tiết có nhất quán không
# (nhãn chi tiết 0 = real phải trùng với label = 1; các nhãn 1–5 là các kiểu tin giả phải trùng với label = 0)
print(pd.crosstab(df["fine-grained label"], df["label"], margins=True))

label                  0     1    All
fine-grained label                   
0                      0  7502   7502
1                   1477     0   1477
2                   1612     0   1612
3                   3444     0   3444
4                   2574     0   2574
5                    300     0    300
All                 9407  7502  16909


In [ ]:
# Ô F2 — Chia dữ liệu theo chủ đề và tiền xử lý thành memmap .npy (khoảng 15–25 phút trên CPU)
import os, shutil, numpy as np, pandas as pd
from PIL import Image
from transformers import CLIPProcessor
COPY_TO_DRIVE = False          # dữ liệu đã có trên Drive; đặt True nếu cần tạo lại bản trên Drive
ROOT, OUT = "/content/finefake", "/content/npy_ff"
DST = "/content/drive/MyDrive/NLP_project/fed/finefake"; os.makedirs(OUT, exist_ok=True)

df = pd.read_pickle(f"{ROOT}/FineFake.pkl")
df = df[df["topic"] != "Uncategorized"].reset_index(drop=True)
df["y"] = (df["label"] == 0).astype(int)                     # gốc: 0=fake, 1=real → y: 1=fake
ID_T, OOD_T = ["Politics", "Business", "Health", "Society"], ["Entertainment", "Conflict"]
rng = np.random.default_rng(0); df["split"] = ""
for t in ID_T:                                               # chia 6:2:2, phân tầng theo chủ đề × nhãn
    for yv in (0, 1):
        idx = df.index[(df.topic == t) & (df.y == yv)].to_numpy(); rng.shuffle(idx)
        a, b = int(0.6 * len(idx)), int(0.8 * len(idx))
        df.loc[idx[:a], "split"], df.loc[idx[a:b], "split"], df.loc[idx[b:], "split"] = "train", "validation", "test"
df.loc[df.topic == "Entertainment", "split"] = "ood_entertainment"
df.loc[df.topic == "Conflict", "split"] = "ood_conflict"
print(pd.crosstab(df.split, df.topic)); print("\nTỉ lệ giả theo tập:\n", df.groupby("split").y.mean().round(3))
df["ext"] = df.image_path.str.lower().str.rsplit(".", n=1).str[-1]
print("\nĐuôi ảnh theo nền tảng:\n", pd.crosstab(df.platform, df.ext))

proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
def load(p):
    try:
        im = Image.open(os.path.join(ROOT, p))
        if getattr(im, "is_animated", False): im.seek(0)
        return im.convert("RGB"), 0
    except Exception:
        return Image.new("RGB", (224, 224), (128, 128, 128)), 1
TOPIC = {t: i for i, t in enumerate(ID_T + OOD_T)}; PLAT = {p: i for i, p in enumerate(sorted(df.platform.unique()))}
for sp in ["train", "validation", "test", "ood_entertainment", "ood_conflict"]:
    d = df[df.split == sp].reset_index(drop=True); n = len(d)
    pix = np.lib.format.open_memmap(f"{OUT}/{sp}_pix.npy", mode="w+", dtype=np.uint8, shape=(n, 3, 224, 224))
    ids, msk, bad = np.zeros((n, 77), np.int64), np.zeros((n, 77), np.int64), 0
    for s in range(0, n, 64):
        b = d.iloc[s:s + 64]; ims = []
        for p in b.image_path:
            im, e = load(p); ims.append(im); bad += e
        px = proc.image_processor(ims, do_rescale=False, do_normalize=False, return_tensors="np")["pixel_values"]
        tk = proc.tokenizer([str(t) for t in b.text], truncation=True, padding="max_length", max_length=77, return_tensors="np")
        k = len(ims); pix[s:s + k] = np.clip(np.round(px), 0, 255).astype(np.uint8)
        ids[s:s + k], msk[s:s + k] = tk["input_ids"], tk["attention_mask"]
    pix.flush(); del pix
    np.save(f"{OUT}/{sp}_ids.npy", ids); np.save(f"{OUT}/{sp}_msk.npy", msk); np.save(f"{OUT}/{sp}_lab.npy", d.y.values.astype(np.int64))
    np.save(f"{OUT}/{sp}_topic.npy", d.topic.map(TOPIC).values); np.save(f"{OUT}/{sp}_plat.npy", d.platform.map(PLAT).values)
    d[["topic", "platform", "y", "image_path"]].to_csv(f"{OUT}/{sp}_meta.csv", index=False)
    print(f"{sp:18s} n={n:5d} | fake={d.y.mean()*100:5.1f}% | ảnh lỗi={bad} | chữ bị cắt ở 77 token={(msk.sum(1) == 77).mean()*100:5.1f}%")

tr, te = set(df[df.split == "train"].text.astype(str)), df[df.split == "test"].text.astype(str)
print("\nChú thích test trùng nguyên văn với train:", int(te.isin(tr).sum()), "/", len(te))
if COPY_TO_DRIVE:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"): drive.mount("/content/drive")
    shutil.copytree(OUT, f"{DST}/npy", dirs_exist_ok=True); print("Đã chép lên Drive:", f"{DST}/npy")
else:
    print("Không chép lên Drive (COPY_TO_DRIVE = False); dữ liệu đã có sẵn ở", f"{DST}/npy")

topic              Business  Conflict  Entertainment  Health  Politics  \
split                                                                    
ood_conflict              0      1718              0       0         0   
ood_entertainment         0         0           3699       0         0   
test                    201         0              0     143      1146   
train                   601         0              0     425      3436   
validation              201         0              0     142      1145   

topic              Society  
split                       
ood_conflict             0  
ood_entertainment        0  
test                   789  
train                 2362  
validation             788  

Tỉ lệ giả theo tập:
 split
ood_conflict         0.497
ood_entertainment    0.589
test                 0.554
train                0.554
validation           0.554
Name: y, dtype: float64

Đuôi ảnh theo nền tảng:
 ext             gif  ico  jpeg   png
platform                    

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/592 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/862k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

... (7 outputs left out)

/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


ood_entertainment  n= 3699 | fake= 58.9% | ảnh lỗi=0 | chữ bị cắt ở 77 token= 18.2%


/usr/local/lib/python3.13/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


ood_conflict       n= 1718 | fake= 49.7% | ảnh lỗi=0 | chữ bị cắt ở 77 token= 27.2%

Chú thích test trùng nguyên văn với train: 189 / 2279
Không chép lên Drive (COPY_TO_DRIVE = False); dữ liệu đã có sẵn ở /content/drive/MyDrive/NLP_project/fed/finefake/npy


## 2. Port the training code to FineFake
Paths, dataset class with CLIP normalisation, topic clients, de-duplicated test views.

In [ ]:
# Ô F3a — Chuyển toàn bộ hàm sang FineFake
import os, shutil, gc, hashlib, numpy as np, torch
from torch.utils.data import Subset
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
FED = "/content/drive/MyDrive/NLP_project/fed/finefake"; RES_PATH = f"{FED}/results_finefake.json"; NPY = "/content/npy_ff"
if not os.path.exists(f"{NPY}/train_lab.npy"): shutil.copytree(f"{FED}/npy", NPY, dirs_exist_ok=True)
MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073]).view(3, 1, 1)
STD = torch.tensor([0.26862954, 0.26130258, 0.27577711]).view(3, 1, 1)
class FF(MM):                                   # ảnh lưu dạng uint8 → chuẩn hóa theo CLIP khi nạp
    def __getitem__(s, i):
        d = MM.__getitem__(s, i); d["pixel_values"] = (d["pixel_values"].float() / 255.0 - MEAN) / STD; return d
SPLITS = ["train", "validation", "test", "ood_entertainment", "ood_conflict"]; TESTS = SPLITS[2:]
enc = {sp: FF(sp) for sp in SPLITS}
ID_T = ["Politics", "Business", "Health", "Society"]
TOP_TR = np.load(f"{NPY}/train_topic.npy")
TOPIC_PARTS = [np.where(TOP_TR == i)[0].tolist() for i in range(4)]
for t, p in zip(ID_T, TOPIC_PARTS): print(f"client {t:9s}: n={len(p):5d} | fake={enc['train'].labels[p].mean()*100:5.1f}%")

h = lambda x: hashlib.md5(np.ascontiguousarray(x).tobytes()).hexdigest()
TR_H = {h(x) for x in enc["train"].ids}
DEDUP = {s: [i for i, x in enumerate(enc[s].ids) if h(x) not in TR_H] for s in TESTS}
for s in TESTS: print(f"{s:18s}: {len(enc[s]) - len(DEDUP[s])} mẫu trùng chữ với train → còn {len(DEDUP[s])}")
x = enc["train"][0]["pixel_values"]; print(f"Kiểm tra chuẩn hóa ảnh: mean={x.mean():.2f}, std={x.std():.2f} (kỳ vọng khoảng 0 và 1)")

def eval_all(mm, name):
    r = {"val": evaluate(mm, enc["validation"])}
    for s in TESTS:
        r[s] = evaluate(mm, enc[s]); r[s + "_dedup"] = evaluate(mm, Subset(enc[s], DEDUP[s]))
    r["OOD"] = {k: float(np.mean([r[s][k] for s in TESTS[1:]])) for k in ("acc", "f1", "auc")}
    print(f"[{name}] val F1 {r['val']['f1']:.2f} | test Acc/F1/AUC {r['test']['acc']:.1f}/{r['test']['f1']:.1f}/{r['test']['auc']:.1f} "
          f"(bỏ trùng: {r['test_dedup']['f1']:.1f}/{r['test_dedup']['auc']:.1f}) | OOD Acc/F1/AUC {r['OOD']['acc']:.1f}/{r['OOD']['f1']:.1f}/{r['OOD']['auc']:.1f}")
    return r

@torch.no_grad()
def probs(mm, dset, bs=64):
    mm.eval(); P, Y = [], []
    for b in loader(dset, bs, False):
        b = batch_to(b)
        with torch.autocast(device, dtype=torch.float16, enabled=(device == "cuda")):
            _, lg = mm(b["input_ids"], b["attention_mask"], b["pixel_values"])
        P.append(torch.softmax(lg.float(), 1)[:, 1].cpu()); Y.append(b["labels"].cpu())
    return torch.cat(P).numpy(), torch.cat(Y).numpy()

def _metr(y, p):
    pred = (p > 0.5).astype(int)
    return dict(acc=float(accuracy_score(y, pred) * 100), f1=float(f1_score(y, pred) * 100), auc=float(roc_auc_score(y, p) * 100))

def late_fusion(path_img, path_txt, TAG):
    SPL = ["validation"] + TESTS; PR = {}
    for mode, path in [("image", path_img), ("text", path_txt)]:
        mm = CLIPModalMFND(mode).to(device); set_state(mm, INIT); set_state(mm, torch.load(path))
        PR[mode] = {s: probs(mm, enc[s]) for s in SPL}; del mm; gc.collect(); torch.cuda.empty_cache()
    res = {}
    for s in SPL:
        y = PR["image"][s][1]; p = (PR["image"][s][0] + PR["text"][s][0]) / 2
        res["val" if s == "validation" else s] = _metr(y, p)
        if s in TESTS: res[s + "_dedup"] = _metr(y[DEDUP[s]], p[DEDUP[s]])
    res["OOD"] = {k: float(np.mean([res[s][k] for s in TESTS[1:]])) for k in ("acc", "f1", "auc")}
    print(f"[{TAG}] val F1 {res['val']['f1']:.2f} | test Acc/F1/AUC {res['test']['acc']:.1f}/{res['test']['f1']:.1f}/{res['test']['auc']:.1f} "
          f"| OOD Acc/F1/AUC {res['OOD']['acc']:.1f}/{res['OOD']['f1']:.1f}/{res['OOD']['auc']:.1f}")
    save_result(TAG, res)

def ff_fed(kind, SEED_RUN=0, ROUNDS=20):
    """FedAvg với 4 client = 4 chủ đề. kind: 'light' (Light-MFND đầy đủ), 'image', 'text'."""
    global m, m2
    if kind == "light":
        model, keys, mb = m, KEYS, 8.27
    else:
        if "m2" in globals(): del m2
        gc.collect(); torch.cuda.empty_cache()
        m2 = CLIPModalMFND(kind).to(device); set_state(m2, INIT); model = m2
        part = "vision_model" if kind == "image" else "text_model"
        keys = [k for k in KEYS if ("lora_" in k and part in k) or k.startswith("classifier")]
        mb = sum(INIT[k].numel() for k in keys) * 4 / 1e6
    name = {"light": "FedSVLM (Light-MFND)", "image": "FedAvg chỉ ảnh", "text": "FedAvg chỉ chữ"}[kind]
    TAG = f"FineFake | {name} (client = chủ đề, seed={SEED_RUN})"; CK = f"{FED}/ck_ff_{kind}_s{SEED_RUN}.pt"
    g, best, best_g, LOG, r0 = _load_ck(CK, {k: INIT[k].clone() for k in keys})
    best_g, LOG = _fedavg_loop(model, keys, TOPIC_PARTS, g, best, best_g, LOG, r0, ROUNDS, SEED_RUN, CK)
    return _finish(model, best_g, LOG, TAG, mb, f"{FED}/ff_{kind}_s{SEED_RUN}_best.pt")
print("Đã chuyển sang FineFake. Kết quả lưu ở:", RES_PATH)

client Politics : n= 3436 | fake= 51.9%
client Business : n=  601 | fake= 52.1%
client Health   : n=  425 | fake= 42.8%
client Society  : n= 2362 | fake= 63.8%
test              : 189 mẫu trùng chữ với train → còn 2090
ood_entertainment : 20 mẫu trùng chữ với train → còn 3679
ood_conflict      : 18 mẫu trùng chữ với train → còn 1700
Kiểm tra chuẩn hóa ảnh: mean=-0.37, std=1.23 (kỳ vọng khoảng 0 và 1)
Đã chuyển sang FineFake. Kết quả lưu ở: /content/drive/MyDrive/NLP_project/fed/finefake/results_finefake.json


In [ ]:
# Ô F3b — Kiểm tra chuẩn hóa ảnh trên nhiều ảnh
xs = torch.stack([enc["train"][i]["pixel_values"] for i in np.random.default_rng(0).choice(len(enc["train"]), 256, replace=False)])
print(f"mean={xs.mean():.2f}, std={xs.std():.2f}, min={xs.min():.2f}, max={xs.max():.2f}")
print("Kỳ vọng: mean trong khoảng −0.5..0.5, std khoảng 0.8..1.2, min ≈ −1.79, max ≈ 2.15")

mean=-0.10, std=1.11, min=-1.79, max=2.15
Kỳ vọng: mean trong khoảng −0.5..0.5, std khoảng 0.8..1.2, min ≈ −1.79, max ≈ 2.15


## 3. Main experiments
Centralized (5 epochs, and 20 epochs with AUC-based selection), FedSVLM, FedSVLM-LF, local-only training, and a TF-IDF + SVM text baseline.

In [ ]:
# Ô F4a — FineFake: Tập trung Light-MFND (5 epoch, chọn theo val F1, như MiRAGeNews) — khoảng 5 phút
run_centralized(m, "FineFake | Tập trung Light-MFND (seed=0)", train_epochs, 0, "ff_cen_s0_best")

epoch 1 loss 0.7186 | val F1 73.83 AUC 73.91 | 56s
epoch 2 loss 0.6846 | val F1 75.14 AUC 74.00 | 111s
epoch 3 loss 0.6288 | val F1 74.53 AUC 74.62 | 164s
epoch 4 loss 0.5954 | val F1 74.51 AUC 75.56 | 218s
[FineFake | Tập trung Light-MFND (seed=0)] val F1 75.14 | test Acc/F1/AUC 64.9/75.5/76.3 (bỏ trùng: 73.2/74.5) | OOD Acc/F1/AUC 61.1/73.2/72.1


{'val': {'acc': 64.41124780316343,
  'f1': 75.13812154696133,
  'auc': np.float64(74.00415576540166)},
 'test': {'acc': 64.94076349275998,
  'f1': 75.54331190694828,
  'auc': np.float64(76.25330421880163)},
 'test_dedup': {'acc': 62.53588516746411,
  'f1': 73.23076923076923,
  'auc': np.float64(74.45682663535054)},
 'ood_entertainment': {'acc': 62.55744795890781,
  'f1': 75.76552930883639,
  'auc': np.float64(78.42701471663194)},
 'ood_entertainment_dedup': {'acc': 62.51698831204131,
  'f1': 75.74318381706244,
  'auc': np.float64(78.31498775096786)},
 'ood_conflict': {'acc': 59.72060535506402,
  'f1': 70.62818336162988,
  'auc': np.float64(65.75605538207998)},
 'ood_conflict_dedup': {'acc': 59.94117647058823,
  'f1': 70.9594882729211,
  'auc': np.float64(65.53707935462172)},
 'OOD': {'acc': 61.13902665698592,
  'f1': 73.19685633523314,
  'auc': 72.09153504935597},
 'time_s': 253.6988034248352,
 'epochs_run': 4}

In [ ]:
# Ô F4b — FineFake: Tập trung 20 epoch, chọn theo val AUC (cùng ngân sách với FedAvg 20 vòng) — khoảng 19 phút
SEED_RUN, EPOCHS = 0, 20
TAG = f"FineFake | Tập trung Light-MFND (20 epoch, chọn AUC, seed={SEED_RUN})"; CK = f"{FED}/ck_ff_cen20_s{SEED_RUN}.pt"
if os.path.exists(CK):
    S = torch.load(CK, weights_only=False); set_state(m, S["st"])
    best, best_state, LOG, e0 = float(S["best"]), S["best_state"], S["LOG"], S["e"]; print(f"Tiếp tục từ epoch {e0+1}")
else:
    set_state(m, INIT); best, best_state, LOG, e0 = -1.0, None, [], 0
torch.manual_seed(1000 * SEED_RUN + e0)
for ep in range(e0 + 1, EPOCHS + 1):
    t0 = time.time(); l = train_epochs(m, enc["train"], epochs=1); v = evaluate(m, enc["validation"])
    LOG.append(dict(epoch=ep, val_f1=float(v["f1"]), val_auc=float(v["auc"]), loss=float(l), sec=float(time.time() - t0)))
    print(f"epoch {ep:2d} | loss {l:.4f} | val F1 {v['f1']:.2f} AUC {v['auc']:.2f} | {time.time()-t0:.0f}s")
    if v["auc"] > best: best, best_state = float(v["auc"]), get_state(m, KEYS)
    torch.save(dict(st=get_state(m, KEYS), best=best, best_state=best_state, LOG=LOG, e=ep), CK)
set_state(m, best_state); res = eval_all(m, TAG)
res.update(epochs_run=len(LOG), best_epoch=int(np.argmax([x["val_auc"] for x in LOG])) + 1,
           time_s=float(sum(x["sec"] for x in LOG)), log=LOG)
save_result(TAG, res); torch.save(best_state, f"{FED}/ff_cen20_s{SEED_RUN}_best.pt")

epoch  1 | loss 0.7186 | val F1 73.83 AUC 73.91 | 69s
epoch  2 | loss 0.6846 | val F1 75.14 AUC 74.00 | 56s
epoch  3 | loss 0.6288 | val F1 74.53 AUC 74.62 | 54s
epoch  4 | loss 0.5954 | val F1 74.51 AUC 75.56 | 54s
epoch  5 | loss 0.5773 | val F1 74.30 AUC 76.30 | 54s
epoch  6 | loss 0.5611 | val F1 73.16 AUC 76.91 | 53s
epoch  7 | loss 0.5472 | val F1 73.10 AUC 77.49 | 53s
epoch  8 | loss 0.5334 | val F1 75.35 AUC 78.14 | 54s
epoch  9 | loss 0.5216 | val F1 75.03 AUC 78.53 | 54s
epoch 10 | loss 0.5111 | val F1 73.34 AUC 78.71 | 54s
epoch 11 | loss 0.4998 | val F1 75.12 AUC 79.07 | 54s
epoch 12 | loss 0.4889 | val F1 76.20 AUC 79.30 | 54s
epoch 13 | loss 0.4775 | val F1 74.77 AUC 79.36 | 53s
epoch 14 | loss 0.4660 | val F1 76.49 AUC 79.60 | 54s
epoch 15 | loss 0.4542 | val F1 74.82 AUC 79.47 | 54s
epoch 16 | loss 0.4446 | val F1 74.67 AUC 79.54 | 54s
epoch 17 | loss 0.4309 | val F1 73.66 AUC 79.35 | 54s
epoch 18 | loss 0.4183 | val F1 73.70 AUC 79.28 | 55s
epoch 19 | loss 0.4075 | val

In [ ]:
# Ô F5 — FineFake: FedSVLM, 4 client = 4 chủ đề, 20 vòng — khoảng 15–20 phút
ff_fed("light")

round  1 | loss 0.7252 | val Acc 60.37 F1 73.36 AUC 69.37 | 58s
round  2 | loss 0.7213 | val Acc 58.57 F1 72.64 AUC 72.51 | 57s
round  3 | loss 0.7158 | val Acc 61.86 F1 74.03 AUC 73.47 | 58s
round  4 | loss 0.7067 | val Acc 63.14 F1 74.58 AUC 73.66 | 56s
round  5 | loss 0.6934 | val Acc 63.27 F1 74.62 AUC 73.66 | 57s
round  6 | loss 0.6767 | val Acc 64.41 F1 75.11 AUC 73.79 | 56s
round  7 | loss 0.6591 | val Acc 65.25 F1 75.40 AUC 73.90 | 56s
round  8 | loss 0.6437 | val Acc 66.04 F1 75.31 AUC 74.10 | 56s
round  9 | loss 0.6313 | val Acc 66.34 F1 74.50 AUC 74.38 | 56s
round 10 | loss 0.6204 | val Acc 66.65 F1 74.02 AUC 74.70 | 56s
round 11 | loss 0.6119 | val Acc 67.49 F1 74.00 AUC 74.98 | 56s
round 12 | loss 0.6045 | val Acc 67.93 F1 73.87 AUC 75.28 | 56s
round 13 | loss 0.5973 | val Acc 67.93 F1 73.97 AUC 75.60 | 55s
round 14 | loss 0.5908 | val Acc 68.37 F1 74.47 AUC 75.86 | 55s
round 15 | loss 0.5858 | val Acc 67.93 F1 73.84 AUC 76.11 | 56s
round 16 | loss 0.5799 | val Acc 68.54 F

'/content/drive/MyDrive/NLP_project/fed/finefake/ff_light_s0_best.pt'

In [ ]:
# Ô F6 — FineFake: FedSVLM-LF (kết hợp muộn trong học liên kết) — khoảng 30 phút
pi = ff_fed("image"); pt = ff_fed("text")
late_fusion(pi, pt, "FineFake | FedSVLM-LF kết hợp muộn (client = chủ đề, seed=0)")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

round  1 | loss 0.6942 | val Acc 45.25 F1 20.13 AUC 51.48 | 44s
round  2 | loss 0.6930 | val Acc 48.07 F1 33.82 AUC 55.47 | 40s
round  3 | loss 0.6916 | val Acc 54.48 F1 52.82 AUC 59.87 | 40s
round  4 | loss 0.6898 | val Acc 60.98 F1 66.72 AUC 63.53 | 40s
round  5 | loss 0.6876 | val Acc 63.36 F1 72.85 AUC 66.32 | 41s
round  6 | loss 0.6846 | val Acc 62.96 F1 73.99 AUC 68.20 | 41s
round  7 | loss 0.6804 | val Acc 63.14 F1 74.51 AUC 69.35 | 41s
round  8 | loss 0.6749 | val Acc 63.14 F1 74.57 AUC 70.04 | 41s
round  9 | loss 0.6687 | val Acc 63.18 F1 74.61 AUC 70.59 | 41s
round 10 | loss 0.6633 | val Acc 63.14 F1 74.55 AUC 71.31 | 40s
round 11 | loss 0.6588 | val Acc 63.18 F1 74.58 AUC 72.03 | 41s
round 12 | loss 0.6541 | val Acc 63.27 F1 74.62 AUC 72.67 | 40s
round 13 | loss 0.6496 | val Acc 64.28 F1 75.11 AUC 73.20 | 40s
round 14 | loss 0.6442 | val Acc 64.41 F1 75.18 AUC 73.63 | 40s
round 15 | loss 0.6395 | val Acc 64.37 F1 75.13 AUC 73.93 | 41s
round 16 | loss 0.6344 | val Acc 64.59 F

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

round  1 | loss 0.6925 | val Acc 46.62 F1 12.65 AUC 62.29 | 40s
round  2 | loss 0.6888 | val Acc 52.20 F1 29.99 AUC 68.78 | 39s
round  3 | loss 0.6840 | val Acc 53.47 F1 34.02 AUC 72.36 | 39s
round  4 | loss 0.6782 | val Acc 55.58 F1 40.14 AUC 74.69 | 40s
round  5 | loss 0.6691 | val Acc 58.04 F1 47.79 AUC 76.24 | 40s
round  6 | loss 0.6518 | val Acc 63.88 F1 60.71 AUC 77.56 | 40s
round  7 | loss 0.6304 | val Acc 70.61 F1 72.50 AUC 78.25 | 40s
round  8 | loss 0.6153 | val Acc 74.56 F1 78.24 AUC 78.52 | 40s
round  9 | loss 0.6033 | val Acc 75.79 F1 79.93 AUC 78.70 | 40s
round 10 | loss 0.5920 | val Acc 75.97 F1 80.36 AUC 78.81 | 40s
round 11 | loss 0.5812 | val Acc 75.75 F1 80.36 AUC 78.93 | 40s
round 12 | loss 0.5707 | val Acc 75.79 F1 80.50 AUC 79.08 | 40s
round 13 | loss 0.5615 | val Acc 75.57 F1 80.37 AUC 79.28 | 40s
round 14 | loss 0.5530 | val Acc 75.44 F1 80.28 AUC 79.53 | 40s
round 15 | loss 0.5455 | val Acc 75.57 F1 80.39 AUC 79.78 | 41s
round 16 | loss 0.5397 | val Acc 75.57 F

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[FineFake | FedSVLM-LF kết hợp muộn (client = chủ đề, seed=0)] val F1 80.62 | test Acc/F1/AUC 74.9/79.9/82.3 | OOD Acc/F1/AUC 71.6/77.7/79.8


In [ ]:
# Ô F7 — FineFake: Local-only (mỗi chủ đề tự học riêng 5 epoch) — khoảng 5 phút
loc = []
for t, p in zip(ID_T, TOPIC_PARTS):
    set_state(m, INIT); torch.manual_seed(0); train_epochs(m, enc["train"], p, epochs=5)
    loc.append(eval_all(m, f"FineFake | Local {t}"))
keys_ = [k for k in loc[0] if isinstance(loc[0][k], dict)]
res = {k: {mt: float(np.mean([r[k][mt] for r in loc])) for mt in ("acc", "f1", "auc")} for k in keys_}
res["clients"] = dict(zip(ID_T, loc)); save_result("FineFake | Local-only (TB 4 chủ đề)", res)
print("TB 4 client: test", {k: round(v, 1) for k, v in res["test"].items()}, "| OOD", {k: round(v, 1) for k, v in res["OOD"].items()})

[FineFake | Local Politics] val F1 73.65 | test Acc/F1/AUC 69.9/75.7/77.3 (bỏ trùng: 73.3/75.4) | OOD Acc/F1/AUC 65.1/72.7/73.1
[FineFake | Local Business] val F1 62.11 | test Acc/F1/AUC 61.4/63.4/65.2 (bỏ trùng: 60.7/64.1) | OOD Acc/F1/AUC 58.8/60.9/62.2
[FineFake | Local Health] val F1 0.00 | test Acc/F1/AUC 44.6/0.0/63.9 (bỏ trùng: 0.0/62.5) | OOD Acc/F1/AUC 45.7/0.0/62.8
[FineFake | Local Society] val F1 74.28 | test Acc/F1/AUC 62.8/74.5/76.0 (bỏ trùng: 72.2/74.5) | OOD Acc/F1/AUC 59.4/72.4/71.8
TB 4 client: test {'acc': 59.7, 'f1': 53.4, 'auc': 70.6} | OOD {'acc': 57.3, 'f1': 51.5, 'auc': 67.5}


In [ ]:
# Ô F8 — FineFake: SVM + TF-IDF (chỉ chữ, chữ giải mã từ token CLIP) — khoảng 1 phút
from transformers import CLIPTokenizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
tok = CLIPTokenizer.from_pretrained(MODEL_NAME)
txt = lambda sp: [tok.decode(i[mk.astype(bool)].tolist(), skip_special_tokens=True) for i, mk in zip(enc[sp].ids, enc[sp].msk)]
vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2))
clf = LinearSVC(C=1.0, class_weight="balanced").fit(vec.fit_transform(txt("train")), enc["train"].labels)
res = {}
for sp in ["validation"] + TESTS:
    s_ = clf.decision_function(vec.transform(txt(sp))); y = enc[sp].labels
    res["val" if sp == "validation" else sp] = dict(acc=float(accuracy_score(y, s_ > 0) * 100),
                                                     f1=float(f1_score(y, s_ > 0) * 100), auc=float(roc_auc_score(y, s_) * 100))
    if sp in TESTS:
        d_ = DEDUP[sp]
        res[sp + "_dedup"] = dict(acc=float(accuracy_score(y[d_], s_[d_] > 0) * 100),
                                  f1=float(f1_score(y[d_], s_[d_] > 0) * 100), auc=float(roc_auc_score(y[d_], s_[d_]) * 100))
res["OOD"] = {k: float(np.mean([res[s][k] for s in TESTS[1:]])) for k in ("acc", "f1", "auc")}
print("[FineFake | SVM + TF-IDF] test", {k: round(v, 1) for k, v in res["test"].items()},
      "| OOD", {k: round(v, 1) for k, v in res["OOD"].items()})
save_result("FineFake | SVM + TF-IDF", res)

[FineFake | SVM + TF-IDF] test {'acc': 72.8, 'f1': 75.3, 'auc': 80.3} | OOD {'acc': 71.1, 'f1': 73.5, 'auc': 77.7}


{'FineFake | Tập trung Light-MFND (seed=0)': {'val': {'acc': 64.41124780316343,
   'f1': 75.13812154696133,
   'auc': 74.00415576540166},
  'test': {'acc': 64.94076349275998,
   'f1': 75.54331190694828,
   'auc': 76.25330421880163},
  'test_dedup': {'acc': 62.53588516746411,
   'f1': 73.23076923076923,
   'auc': 74.45682663535054},
  'ood_entertainment': {'acc': 62.55744795890781,
   'f1': 75.76552930883639,
   'auc': 78.42701471663194},
  'ood_entertainment_dedup': {'acc': 62.51698831204131,
   'f1': 75.74318381706244,
   'auc': 78.31498775096786},
  'ood_conflict': {'acc': 59.72060535506402,
   'f1': 70.62818336162988,
   'auc': 65.75605538207998},
  'ood_conflict_dedup': {'acc': 59.94117647058823,
   'f1': 70.9594882729211,
   'auc': 65.53707935462172},
  'OOD': {'acc': 61.13902665698592,
   'f1': 73.19685633523314,
   'auc': 72.09153504935597},
  'time_s': 253.6988034248352,
  'epochs_run': 4},
 'FineFake | Tập trung Light-MFND (20 epoch, chọn AUC, seed=0)': {'val': {'acc': 72.1880

## 4. Summary tables

In [ ]:
# Ô F9 — Bảng tổng hợp FineFake (chỉ đọc results_finefake.json)
import json, pandas as pd
R = json.load(open(RES_PATH))
COLS = [("test", "test"), ("test_dedup", "test (bỏ trùng)"), ("ood_entertainment", "OOD Ent."),
        ("ood_conflict", "OOD Conf."), ("OOD", "OOD TB")]
rows = []
for name, r in R.items():
    row = {"Phương pháp": name.replace("FineFake | ", "")}
    for k, lab in COLS:
        if k in r: row[f"{lab} Acc/F1/AUC"] = f"{r[k]['acc']:.1f}/{r[k]['f1']:.1f}/{r[k]['auc']:.1f}"
    row["MB/lượt"] = round(r["update_mb"], 2) if "update_mb" in r else "–"
    row["Vòng/epoch"] = r.get("rounds_run", r.get("epochs_run", "–"))
    rows.append(row)
T = pd.DataFrame(rows); pd.set_option("display.width", 300); print(T.to_string(index=False))
T.to_csv(f"{FED}/bang_tong_hop_finefake.csv", index=False, encoding="utf-8-sig"); print("\nĐã lưu bảng:", f"{FED}/bang_tong_hop_finefake.csv")

                                      Phương pháp test Acc/F1/AUC test (bỏ trùng) Acc/F1/AUC OOD Ent. Acc/F1/AUC OOD Conf. Acc/F1/AUC OOD TB Acc/F1/AUC MB/lượt Vòng/epoch
                    Tập trung Light-MFND (seed=0)  64.9/75.5/76.3             62.5/73.2/74.5      62.6/75.8/78.4       59.7/70.6/65.8    61.1/73.2/72.1       –          4
Tập trung Light-MFND (20 epoch, chọn AUC, seed=0)  73.3/77.5/80.8             71.5/75.1/79.1      75.7/81.5/83.3       64.0/69.1/71.0    69.8/75.3/77.1       –         20
   FedSVLM (Light-MFND) (client = chủ đề, seed=0)  71.6/76.1/79.3             69.9/73.7/77.6      73.4/80.1/82.5       62.4/67.2/69.7    67.9/73.7/76.1    8.27         20
                         Local-only (TB 4 chủ đề)  59.7/53.4/70.6             59.1/51.6/69.1      58.5/55.6/71.2       56.0/47.4/63.7    57.3/51.5/67.5       –          –
                                     SVM + TF-IDF  72.8/75.3/80.3             70.7/72.1/77.7      77.2/81.2/86.0       65.1/65.8/69.4    71.1/73.

In [ ]:
# Ô F10 — In lại kết quả mọi lần chạy FineFake (kèm nhật ký từng vòng/epoch nếu có)
for name, r in json.load(open(RES_PATH)).items():
    print("=" * 100); print(name)
    for x in r.get("log", []):
        tag = f"round {x['round']:2d}" if "round" in x else f"epoch {x['epoch']:2d}"
        print(f"  {tag} | loss {x['loss']:.4f} | val F1 {x['val_f1']:.2f} AUC {x['val_auc']:.2f}")
    t, o = r.get("test", {}), r.get("OOD", {})
    if t and o:
        print(f"  => test Acc/F1/AUC {t['acc']:.1f}/{t['f1']:.1f}/{t['auc']:.1f} | OOD Acc/F1/AUC {o['acc']:.1f}/{o['f1']:.1f}/{o['auc']:.1f}")

FineFake | Tập trung Light-MFND (seed=0)
  => test Acc/F1/AUC 64.9/75.5/76.3 | OOD Acc/F1/AUC 61.1/73.2/72.1
FineFake | Tập trung Light-MFND (20 epoch, chọn AUC, seed=0)
  epoch  1 | loss 0.7186 | val F1 73.83 AUC 73.91
  epoch  2 | loss 0.6846 | val F1 75.14 AUC 74.00
  epoch  3 | loss 0.6288 | val F1 74.53 AUC 74.62
  epoch  4 | loss 0.5954 | val F1 74.51 AUC 75.56
  epoch  5 | loss 0.5773 | val F1 74.30 AUC 76.30
  epoch  6 | loss 0.5611 | val F1 73.16 AUC 76.91
  epoch  7 | loss 0.5472 | val F1 73.10 AUC 77.49
  epoch  8 | loss 0.5334 | val F1 75.35 AUC 78.14
  epoch  9 | loss 0.5216 | val F1 75.03 AUC 78.53
  epoch 10 | loss 0.5111 | val F1 73.34 AUC 78.71
  epoch 11 | loss 0.4998 | val F1 75.12 AUC 79.07
  epoch 12 | loss 0.4889 | val F1 76.20 AUC 79.30
  epoch 13 | loss 0.4775 | val F1 74.77 AUC 79.36
  epoch 14 | loss 0.4660 | val F1 76.49 AUC 79.60
  epoch 15 | loss 0.4542 | val F1 74.82 AUC 79.47
  epoch 16 | loss 0.4446 | val F1 74.67 AUC 79.54
  epoch 17 | loss 0.4309 | val

## 5. Source-shortcut analysis
A classifier that sees only the news source, and AUC computed within each source.

In [ ]:
# Ô F11 — Kiểm tra lối tắt: chỉ dùng NGUỒN TIN để đoán nhãn (không chữ, không ảnh)
P_tr, y_tr = np.load(f"{NPY}/train_plat.npy"), enc["train"].labels
rate = {p: y_tr[P_tr == p].mean() for p in np.unique(P_tr)}          # tỉ lệ giả của từng nguồn, học từ train
res = {}
for s in ["validation"] + TESTS:
    P, y = np.load(f"{NPY}/{s}_plat.npy"), enc[s].labels
    score = np.array([rate.get(p, y_tr.mean()) for p in P]); res[s] = float(roc_auc_score(y, score) * 100)
    print(f"{s:18s}: AUC chỉ dựa vào nguồn tin = {res[s]:.1f}")
save_result("FineFake | Đối chứng: chỉ dùng nguồn tin (platform prior)", {"auc": res})

validation        : AUC chỉ dựa vào nguồn tin = 73.5
test              : AUC chỉ dựa vào nguồn tin = 74.0
ood_entertainment : AUC chỉ dựa vào nguồn tin = 76.7
ood_conflict      : AUC chỉ dựa vào nguồn tin = 71.1


{'FineFake | Tập trung Light-MFND (seed=0)': {'val': {'acc': 64.41124780316343,
   'f1': 75.13812154696133,
   'auc': 74.00415576540166},
  'test': {'acc': 64.94076349275998,
   'f1': 75.54331190694828,
   'auc': 76.25330421880163},
  'test_dedup': {'acc': 62.53588516746411,
   'f1': 73.23076923076923,
   'auc': 74.45682663535054},
  'ood_entertainment': {'acc': 62.55744795890781,
   'f1': 75.76552930883639,
   'auc': 78.42701471663194},
  'ood_entertainment_dedup': {'acc': 62.51698831204131,
   'f1': 75.74318381706244,
   'auc': 78.31498775096786},
  'ood_conflict': {'acc': 59.72060535506402,
   'f1': 70.62818336162988,
   'auc': 65.75605538207998},
  'ood_conflict_dedup': {'acc': 59.94117647058823,
   'f1': 70.9594882729211,
   'auc': 65.53707935462172},
  'OOD': {'acc': 61.13902665698592,
   'f1': 73.19685633523314,
   'auc': 72.09153504935597},
  'time_s': 253.6988034248352,
  'epochs_run': 4},
 'FineFake | Tập trung Light-MFND (20 epoch, chọn AUC, seed=0)': {'val': {'acc': 72.1880

In [ ]:
# Ô F13 — AUC bên trong từng nguồn tin (loại bỏ lối tắt "nhận ra nguồn")
import pandas as pd
EV = ["test", "ood_entertainment", "ood_conflict"]
PL = {s: np.load(f"{NPY}/{s}_plat.npy") for s in EV}
NAMES = sorted(pd.concat([pd.read_csv(f"{NPY}/{s}_meta.csv") for s in SPLITS]).platform.unique())   # đúng thứ tự mã nguồn ở F2
def sc_light(path):
    set_state(m, INIT); set_state(m, torch.load(path)); return {s: probs(m, enc[s]) for s in EV}
def sc_lf(pi, pt):
    out = {}
    for mode, path in [("image", pi), ("text", pt)]:
        mm = CLIPModalMFND(mode).to(device); set_state(mm, INIT); set_state(mm, torch.load(path))
        out[mode] = {s: probs(mm, enc[s]) for s in EV}; del mm; gc.collect(); torch.cuda.empty_cache()
    return {s: ((out["image"][s][0] + out["text"][s][0]) / 2, out["image"][s][1]) for s in EV}
MODELS = {"Tập trung Light-MFND (20 ep)": sc_light(f"{FED}/ff_cen20_s0_best.pt"),
          "FedSVLM": sc_light(f"{FED}/ff_light_s0_best.pt"),
          "FedSVLM-LF": sc_lf(f"{FED}/ff_image_s0_best.pt", f"{FED}/ff_text_s0_best.pt")}
rows, out_json = [], {}
for name, S in MODELS.items():
    p = np.concatenate([S[s][0] for s in EV]); y = np.concatenate([S[s][1] for s in EV]); pl = np.concatenate([PL[s] for s in EV])
    row, per = {"Mô hình": name, "AUC gộp": round(roc_auc_score(y, p) * 100, 1)}, []
    for k in np.unique(pl):
        mk = pl == k
        if mk.sum() >= 50 and 0 < y[mk].mean() < 1:
            a = roc_auc_score(y[mk], p[mk]) * 100; row[f"{NAMES[k]} (n={mk.sum()})"] = round(a, 1); per.append(a)
    row["TB trong từng nguồn"] = round(float(np.mean(per)), 1); rows.append(row); out_json[name] = row
print("Gộp test + 2 tập OOD; chỉ tính nguồn có ≥ 50 mẫu và có cả hai nhãn")
print(pd.DataFrame(rows).to_string(index=False))
save_result("FineFake | Phân tích: AUC trong từng nguồn tin", out_json)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Gộp test + 2 tập OOD; chỉ tính nguồn có ≥ 50 mẫu và có cả hai nhãn
                     Mô hình  AUC gộp  apnews (n=314)  cnn (n=995)  reddit (n=2129)  snope (n=3420)  twitter (n=375)  washingtonpost (n=373)  TB trong từng nguồn
Tập trung Light-MFND (20 ep)     80.0            53.9         49.2             87.0            60.8             52.3                    54.5                 59.6
                     FedSVLM     78.8            54.9         43.3             88.8            59.2             52.6                    53.2                 58.7
                  FedSVLM-LF     82.5            51.9         43.6             94.6            58.3             54.7                    51.6                 59.1


{'FineFake | Tập trung Light-MFND (seed=0)': {'val': {'acc': 64.41124780316343,
   'f1': 75.13812154696133,
   'auc': 74.00415576540166},
  'test': {'acc': 64.94076349275998,
   'f1': 75.54331190694828,
   'auc': 76.25330421880163},
  'test_dedup': {'acc': 62.53588516746411,
   'f1': 73.23076923076923,
   'auc': 74.45682663535054},
  'ood_entertainment': {'acc': 62.55744795890781,
   'f1': 75.76552930883639,
   'auc': 78.42701471663194},
  'ood_entertainment_dedup': {'acc': 62.51698831204131,
   'f1': 75.74318381706244,
   'auc': 78.31498775096786},
  'ood_conflict': {'acc': 59.72060535506402,
   'f1': 70.62818336162988,
   'auc': 65.75605538207998},
  'ood_conflict_dedup': {'acc': 59.94117647058823,
   'f1': 70.9594882729211,
   'auc': 65.53707935462172},
  'OOD': {'acc': 61.13902665698592,
   'f1': 73.19685633523314,
   'auc': 72.09153504935597},
  'time_s': 253.6988034248352,
  'epochs_run': 4},
 'FineFake | Tập trung Light-MFND (20 epoch, chọn AUC, seed=0)': {'val': {'acc': 72.1880

## 6. External knowledge (FineFake `knowledge_embedding`)
A knowledge-only branch trained with FedAvg, and late fusion with the image and text branches (fusion weight chosen on validation).

In [ ]:
# Ô K0 — Tải lại FineFake và chỉ giải nén file .pkl (không cần ảnh)
!pip install -q gdown
import os, zipfile, gdown
ROOT = "/content/finefake"; os.makedirs(ROOT, exist_ok=True)
if not os.path.exists(f"{ROOT}/FineFake.pkl"):
    z = f"{ROOT}/FineFake.zip"
    if not os.path.exists(z): gdown.download(id="16D9ix7ZOisa4VVBznBTBcv1N7TA-jodH", output=z, quiet=False)
    with zipfile.ZipFile(z) as zf:
        name = [n for n in zf.namelist() if n.lower().endswith(".pkl")][0]
        with zf.open(name) as src, open(f"{ROOT}/FineFake.pkl", "wb") as dst: dst.write(src.read())
    os.remove(z)                                     # xóa file zip cho nhẹ máy
print("Đã có:", f"{ROOT}/FineFake.pkl", f"{os.path.getsize(f'{ROOT}/FineFake.pkl')/1e6:.0f} MB")

Downloading...
From (original): https://drive.google.com/uc?id=16D9ix7ZOisa4VVBznBTBcv1N7TA-jodH
From (redirected): https://drive.google.com/uc?id=16D9ix7ZOisa4VVBznBTBcv1N7TA-jodH&confirm=t&uuid=9bdb645d-8b39-48e4-8372-2ed8decba662
To: /content/finefake/FineFake.zip
100%|██████████| 5.06G/5.06G [01:25<00:00, 59.3MB/s]


Đã có: /content/finefake/FineFake.pkl 187 MB


In [ ]:
# Ô K1 — Xem các cột tri thức của FineFake chứa gì
import numpy as np, pandas as pd
df = pd.read_pickle("/content/finefake/FineFake.pkl")
df = df[df["topic"] != "Uncategorized"]
for c in ["entity_id", "knowledge_embedding", "description", "relation"]:
    v = df[c].iloc[0]
    print(f"\n=== {c} | kiểu: {type(v).__name__}")
    if isinstance(v, (list, tuple)): print("độ dài:", len(v), "| phần tử đầu:", type(v[0]).__name__ if len(v) else None, str(v[:2])[:300])
    elif isinstance(v, np.ndarray): print("shape:", v.shape, "| dtype:", v.dtype)
    else: print(str(v)[:300])
    empty = df[c].apply(lambda x: x is None or (hasattr(x, "__len__") and len(x) == 0) or (isinstance(x, float) and np.isnan(x)))
    print(f"tỉ lệ rỗng: {empty.mean()*100:.1f}%")
    print("tỉ lệ rỗng theo nền tảng (%):", df.assign(e=empty).groupby("platform").e.mean().mul(100).round(1).to_dict())
lens = df["knowledge_embedding"].apply(lambda x: len(x) if hasattr(x, "__len__") else 0)
print("\nSố vectơ tri thức mỗi mẫu: min", lens.min(), "| TB", round(lens.mean(), 1), "| max", lens.max())


=== entity_id | kiểu: list
độ dài: 0 | phần tử đầu: None []
tỉ lệ rỗng: 21.6%
tỉ lệ rỗng theo nền tảng (%): {'apnews': 2.0, 'cdc_gov': 24.5, 'cnn': 15.4, 'reddit': 49.2, 'snope': 15.4, 'twitter': 0.1, 'washingtonpost': 4.6}

=== knowledge_embedding | kiểu: ndarray
shape: (100,) | dtype: float64
tỉ lệ rỗng: 0.0%
tỉ lệ rỗng theo nền tảng (%): {'apnews': 0.0, 'cdc_gov': 0.0, 'cnn': 0.0, 'reddit': 0.0, 'snope': 0.0, 'twitter': 0.0, 'washingtonpost': 0.0}

=== description | kiểu: list
độ dài: 0 | phần tử đầu: None []
tỉ lệ rỗng: 21.3%
tỉ lệ rỗng theo nền tảng (%): {'apnews': 2.0, 'cdc_gov': 24.5, 'cnn': 15.4, 'reddit': 47.9, 'snope': 15.4, 'twitter': 0.1, 'washingtonpost': 4.6}

=== relation | kiểu: list
độ dài: 0 | phần tử đầu: None []
tỉ lệ rỗng: 21.3%
tỉ lệ rỗng theo nền tảng (%): {'apnews': 2.0, 'cdc_gov': 24.5, 'cnn': 15.4, 'reddit': 47.9, 'snope': 15.4, 'twitter': 0.1, 'washingtonpost': 4.6}

Số vectơ tri thức mỗi mẫu: min 100 | TB 100.0 | max 100


In [ ]:
# Ô K2 — Nhánh tri thức (knowledge_embedding 100 chiều) + kết hợp muộn 3 nhánh; đánh giá cả trong từng nguồn
import numpy as np, pandas as pd, torch, torch.nn as nn, copy, gc
from sklearn.metrics import roc_auc_score
raw = pd.read_pickle("/content/finefake/FineFake.pkl")
raw = raw[raw["topic"] != "Uncategorized"].reset_index(drop=True); raw["y"] = (raw["label"] == 0).astype(int)
rng = np.random.default_rng(0); raw["split"] = ""                      # chia y hệt ô F2
for t in ["Politics", "Business", "Health", "Society"]:
    for yv in (0, 1):
        idx = raw.index[(raw.topic == t) & (raw.y == yv)].to_numpy(); rng.shuffle(idx)
        a, b = int(0.6 * len(idx)), int(0.8 * len(idx))
        raw.loc[idx[:a], "split"], raw.loc[idx[a:b], "split"], raw.loc[idx[b:], "split"] = "train", "validation", "test"
raw.loc[raw.topic == "Entertainment", "split"] = "ood_entertainment"; raw.loc[raw.topic == "Conflict", "split"] = "ood_conflict"
KX, ok = {}, True
for s in SPLITS:
    d = raw[raw.split == s].reset_index(drop=True); meta = pd.read_csv(f"{NPY}/{s}_meta.csv")
    ok &= bool((d.y.values == meta.y.values).all() and (d.image_path.values == meta.image_path.values).all())
    KX[s] = np.stack(d.knowledge_embedding.values).astype(np.float32)
print("Thứ tự mẫu khớp với dữ liệu đã xử lý:", ok); assert ok

NAMES = sorted(pd.concat([pd.read_csv(f"{NPY}/{s}_meta.csv") for s in SPLITS]).platform.unique())
PLs = {s: np.load(f"{NPY}/{s}_plat.npy") for s in SPLITS}
z = np.abs(KX["train"]).sum(1) == 0
print(f"Vectơ tri thức toàn 0 (train): {z.mean()*100:.1f}% | theo nguồn (%):",
      {NAMES[k]: round(z[PLs['train'] == k].mean() * 100, 1) for k in np.unique(PLs["train"])})

mu, sd = KX["train"].mean(0), KX["train"].std(0) + 1e-6              # chuẩn hóa theo train (đơn giản hóa, xem lưu ý)
Xn = {s: torch.tensor((KX[s] - mu) / sd) for s in SPLITS}; Yt = {s: torch.tensor(enc[s].labels) for s in SPLITS}
mlp = lambda: nn.Sequential(nn.Linear(100, 64), nn.ReLU(), nn.Dropout(0.1), nn.Linear(64, 2))
def local(state, idx):
    net = mlp(); net.load_state_dict(state); opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=0.01)
    X, y = Xn["train"][idx], Yt["train"][idx]; perm = torch.randperm(len(idx))
    for i in range(0, len(idx), 64):
        b = perm[i:i + 64]; loss = nn.functional.cross_entropy(net(X[b]), y[b]); opt.zero_grad(); loss.backward(); opt.step()
    return net.state_dict()
def kprob(state, s):
    net = mlp(); net.load_state_dict(state); net.eval()
    with torch.no_grad(): return torch.softmax(net(Xn[s]), 1)[:, 1].numpy()
torch.manual_seed(0); gk = mlp().state_dict(); best, best_k = -1, None; N = sum(len(p) for p in TOPIC_PARTS)
for r in range(1, 21):                                                   # FedAvg 20 vòng, 4 client = 4 chủ đề
    sts = [local(gk, torch.tensor(p)) for p in TOPIC_PARTS]
    gk = {k: sum(st[k] * (len(p) / N) for st, p in zip(sts, TOPIC_PARTS)) for k in gk}
    va = roc_auc_score(Yt["validation"].numpy(), kprob(gk, "validation")) * 100
    if va > best: best, best_k = va, copy.deepcopy(gk)
    if r % 5 == 0: print(f"round {r:2d} | val AUC nhánh tri thức {va:.2f}")

EVS = ["validation"] + TESTS
def modal_probs(mode, path):
    mm = CLIPModalMFND(mode).to(device); set_state(mm, INIT); set_state(mm, torch.load(path))
    out = {s: probs(mm, enc[s])[0] for s in EVS}; del mm; gc.collect(); torch.cuda.empty_cache(); return out
PI, PT = modal_probs("image", f"{FED}/ff_image_s0_best.pt"), modal_probs("text", f"{FED}/ff_text_s0_best.pt")
PK = {s: kprob(best_k, s) for s in EVS}
def within(P):
    p = np.concatenate([P[s] for s in TESTS]); y = np.concatenate([enc[s].labels for s in TESTS]); pl = np.concatenate([PLs[s] for s in TESTS])
    per = {NAMES[k]: roc_auc_score(y[pl == k], p[pl == k]) * 100 for k in np.unique(pl) if (pl == k).sum() >= 50 and 0 < y[pl == k].mean() < 1}
    return roc_auc_score(y, p) * 100, float(np.mean(list(per.values()))), per
COMBOS = {"Chỉ tri thức (FedAvg MLP)": PK,
          "FedSVLM-LF (ảnh + chữ)": {s: (PI[s] + PT[s]) / 2 for s in EVS},
          "FedSVLM-LF + tri thức (3 nhánh)": {s: (PI[s] + PT[s] + PK[s]) / 3 for s in EVS}}
rows, out = [], {}
for name, P in COMBOS.items():
    g, w, per = within(P)
    row = {"Phương pháp": name, "test AUC": round(roc_auc_score(enc["test"].labels, P["test"]) * 100, 1),
           "OOD AUC": round(float(np.mean([roc_auc_score(enc[s].labels, P[s]) * 100 for s in TESTS[1:]])), 1),
           "AUC gộp": round(g, 1), "TB trong từng nguồn": round(w, 1)}
    row.update({k: round(v, 1) for k, v in per.items()}); rows.append(row); out[name] = row
pd.set_option("display.width", 300); print(pd.DataFrame(rows).to_string(index=False))
save_result("FineFake | Thử nghiệm nhánh tri thức (knowledge_embedding)", out)

Thứ tự mẫu khớp với dữ liệu đã xử lý: True
Vectơ tri thức toàn 0 (train): 24.3% | theo nguồn (%): {'apnews': np.float64(3.8), 'cdc_gov': np.float64(27.0), 'cnn': np.float64(19.5), 'reddit': np.float64(56.3), 'snope': np.float64(19.1), 'twitter': np.float64(0.2), 'washingtonpost': np.float64(5.7)}
round  5 | val AUC nhánh tri thức 62.71
round 10 | val AUC nhánh tri thức 63.68
round 15 | val AUC nhánh tri thức 64.98
round 20 | val AUC nhánh tri thức 64.94


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

                    Phương pháp  test AUC  OOD AUC  AUC gộp  TB trong từng nguồn  apnews  cnn  reddit  snope  twitter  washingtonpost
      Chỉ tri thức (FedAvg MLP)      66.1     63.4     65.7                 52.6    38.6 50.2    67.4   53.8     55.7            49.9
         FedSVLM-LF (ảnh + chữ)      82.3     79.8     82.5                 59.1    51.9 43.6    94.6   58.3     54.7            51.6
FedSVLM-LF + tri thức (3 nhánh)      80.9     77.3     80.3                 57.4    43.1 46.3    89.3   56.5     56.8            52.2


{'FineFake | Tập trung Light-MFND (seed=0)': {'val': {'acc': 64.41124780316343,
   'f1': 75.13812154696133,
   'auc': 74.00415576540166},
  'test': {'acc': 64.94076349275998,
   'f1': 75.54331190694828,
   'auc': 76.25330421880163},
  'test_dedup': {'acc': 62.53588516746411,
   'f1': 73.23076923076923,
   'auc': 74.45682663535054},
  'ood_entertainment': {'acc': 62.55744795890781,
   'f1': 75.76552930883639,
   'auc': 78.42701471663194},
  'ood_entertainment_dedup': {'acc': 62.51698831204131,
   'f1': 75.74318381706244,
   'auc': 78.31498775096786},
  'ood_conflict': {'acc': 59.72060535506402,
   'f1': 70.62818336162988,
   'auc': 65.75605538207998},
  'ood_conflict_dedup': {'acc': 59.94117647058823,
   'f1': 70.9594882729211,
   'auc': 65.53707935462172},
  'OOD': {'acc': 61.13902665698592,
   'f1': 73.19685633523314,
   'auc': 72.09153504935597},
  'time_s': 253.6988034248352,
  'epochs_run': 4},
 'FineFake | Tập trung Light-MFND (20 epoch, chọn AUC, seed=0)': {'val': {'acc': 72.1880

In [ ]:
# Ô K3 — Ghép nhánh tri thức với các nhánh khác (không huấn luyện lại; trọng số chọn trên VALIDATION)
set_state(m, INIT); set_state(m, torch.load(f"{FED}/ff_light_s0_best.pt"))
PL_ = {s: probs(m, enc[s])[0] for s in EVS}                          # FedSVLM (Light-MFND) để ghép thử với tri thức
avg = lambda *Ps: {s: np.mean([P[s] for P in Ps], 0) for s in EVS}
COMBOS = {"Ảnh + chữ (FedSVLM-LF, mốc)": avg(PI, PT),
          "Chữ + tri thức": avg(PT, PK),
          "Ảnh + tri thức": avg(PI, PK),
          "Ảnh + chữ + tri thức (đều nhau)": avg(PI, PT, PK),
          "FedSVLM + tri thức": avg(PL_, PK)}
va_y = enc["validation"].labels; best_w, best_va = 0.0, -1
for w in np.arange(0, 0.55, 0.05):                                  # trọng số của tri thức, chọn theo val AUC
    p = (1 - w) * (PI["validation"] + PT["validation"]) / 2 + w * PK["validation"]
    a = roc_auc_score(va_y, p)
    if a > best_va: best_va, best_w = a, w
COMBOS[f"Ảnh + chữ + tri thức (trọng số tri thức = {best_w:.2f}, chọn trên val)"] = {
    s: (1 - best_w) * (PI[s] + PT[s]) / 2 + best_w * PK[s] for s in EVS}
rows, out = [], {}
for name, P in COMBOS.items():
    g, w_, per = within(P)
    row = {"Tổ hợp": name, "val AUC": round(roc_auc_score(va_y, P["validation"]) * 100, 1),
           "test AUC": round(roc_auc_score(enc["test"].labels, P["test"]) * 100, 1),
           "OOD AUC": round(float(np.mean([roc_auc_score(enc[s].labels, P[s]) * 100 for s in TESTS[1:]])), 1),
           "AUC gộp": round(g, 1), "TB trong từng nguồn": round(w_, 1)}
    rows.append(row); out[name] = row
pd.set_option("display.width", 300); print(pd.DataFrame(rows).to_string(index=False))
save_result("FineFake | Thử nghiệm ghép tri thức với các nhánh khác", out)

                                                        Tổ hợp  val AUC  test AUC  OOD AUC  AUC gộp  TB trong từng nguồn
                                   Ảnh + chữ (FedSVLM-LF, mốc)     82.0      82.3     79.8     82.5                 59.1
                                                Chữ + tri thức     78.8      79.0     75.5     78.6                 56.0
                                                Ảnh + tri thức     74.0      75.5     71.7     74.4                 55.4
                               Ảnh + chữ + tri thức (đều nhau)     80.1      80.9     77.3     80.3                 57.4
                                            FedSVLM + tri thức     77.8      79.3     75.7     78.7                 58.1
Ảnh + chữ + tri thức (trọng số tri thức = 0.05, chọn trên val)     82.0      82.6     79.9     82.7                 58.9


{'FineFake | Tập trung Light-MFND (seed=0)': {'val': {'acc': 64.41124780316343,
   'f1': 75.13812154696133,
   'auc': 74.00415576540166},
  'test': {'acc': 64.94076349275998,
   'f1': 75.54331190694828,
   'auc': 76.25330421880163},
  'test_dedup': {'acc': 62.53588516746411,
   'f1': 73.23076923076923,
   'auc': 74.45682663535054},
  'ood_entertainment': {'acc': 62.55744795890781,
   'f1': 75.76552930883639,
   'auc': 78.42701471663194},
  'ood_entertainment_dedup': {'acc': 62.51698831204131,
   'f1': 75.74318381706244,
   'auc': 78.31498775096786},
  'ood_conflict': {'acc': 59.72060535506402,
   'f1': 70.62818336162988,
   'auc': 65.75605538207998},
  'ood_conflict_dedup': {'acc': 59.94117647058823,
   'f1': 70.9594882729211,
   'auc': 65.53707935462172},
  'OOD': {'acc': 61.13902665698592,
   'f1': 73.19685633523314,
   'auc': 72.09153504935597},
  'time_s': 253.6988034248352,
  'epochs_run': 4},
 'FineFake | Tập trung Light-MFND (20 epoch, chọn AUC, seed=0)': {'val': {'acc': 72.1880

## 7. Source-balanced client resampling
Each client resamples its data so that every source has equally many real and fake items. Two runs (seed 0 and 1) against the standard FedSVLM-LF.

In [ ]:
# Ô B0 — Huấn luyện CÂN BẰNG THEO NGUỒN: chuẩn bị
import pandas as pd
import inspect, os, time
EVS = EVS if "EVS" in globals() else ["validation"] + TESTS
Ytr = np.asarray(enc["train"].labels)
_pp = [p for p in (f"{NPY}/train_plat.npy", f"{FED}/npy/train_plat.npy") if os.path.exists(p)]
PLAT = np.load(_pp[0], allow_pickle=True).astype(str); assert len(PLAT) == len(Ytr), (len(PLAT), len(Ytr))
PARTS = list(TOPIC_PARTS.values()) if isinstance(TOPIC_PARTS, dict) else list(TOPIC_PARTS)
PARTS = [np.asarray(p) for p in PARTS]; MIN_PER = 5     # nguồn có < 5 mẫu ở nhãn ít hơn thì bỏ (không cân bằng được)

def bal_idx(part, rng, verbose=False):
    y, s, n = Ytr[part], PLAT[part], len(part); groups, dropped = [], []
    for src in np.unique(s):
        msk = s == src; i0, i1 = part[msk & (y == 0)], part[msk & (y == 1)]
        if min(len(i0), len(i1)) < MIN_PER: dropped.append(f"{src}({len(i0)}/{len(i1)})"); continue
        groups.append((src, i0, i1, msk.sum()))
    tot = sum(g[3] for g in groups); out = []
    for src, i0, i1, c in groups:
        h = max(1, int(round(n * c / tot / 2)))
        out += list(rng.choice(i0, h, replace=True)) + list(rng.choice(i1, h, replace=True))
    if verbose:
        print(f"  n={n} | giữ {[g[0] for g in groups]} ({tot/n*100:.0f}% dữ liệu gốc) | bỏ (fake/real) {dropped}")
    return out

print("Tương quan nguồn–nhãn trong tập train (tỉ lệ nhãn 1 = real):")
print(pd.crosstab(PLAT, Ytr, normalize="index").round(2).assign(n=pd.Series(PLAT).value_counts()))
for k, p in enumerate(PARTS): print(f"client {k}:"); bal_idx(p, np.random.default_rng(0), verbose=True)

_ps = list(inspect.signature(CLIPModalMFND.__init__).parameters)[1:]; print("Tham số CLIPModalMFND:", _ps)
_kw = next((p for p in _ps if p in ("mode", "modality", "kind", "branch", "which")), None)
def build_modal(kind):
    torch.manual_seed(0); mod = (CLIPModalMFND(**{_kw: kind}) if _kw else CLIPModalMFND(kind)).to(device)
    keys = [n for n, p in mod.named_parameters() if p.requires_grad]
    set_state(mod, {k: v for k, v in INIT.items() if k in keys}); return mod, keys

def fed_bal(kind, ROUNDS=20, SAVE_EVERY=5):
    mod, keys = build_modal(kind); CK = f"{FED}/ck_ff_bal_{kind}.pt"; BEST = f"{FED}/ff_bal_{kind}_s0_best.pt"
    S = torch.load(CK, weights_only=False) if os.path.exists(CK) else dict(g=get_state(mod, keys), best=-1.0, best_g=None, LOG=[], r=0)
    if S["r"]: print(f"Tiếp tục từ checkpoint, vòng {S['r']+1}")
    N = sum(len(p) for p in PARTS)
    for r in range(S["r"] + 1, ROUNDS + 1):
        t0 = time.time(); g = S["g"]; acc = {k: torch.zeros_like(v) for k, v in g.items()}; L = []
        for k, part in enumerate(PARTS):
            set_state(mod, g); L.append(train_epochs(mod, enc["train"], bal_idx(part, np.random.default_rng(1000 * r + k)), epochs=1))
            st = get_state(mod, keys); w = len(part) / N
            for key in acc: acc[key] += st[key] * w
        set_state(mod, acc); v = evaluate(mod, enc["validation"]); S["g"], S["r"] = acc, r
        S["LOG"].append(dict(round=r, val_auc=float(v["auc"]), val_f1=float(v["f1"]), loss=float(np.mean(L)), sec=time.time() - t0))
        print(f"[{kind}] round {r:2d} | loss {np.mean(L):.4f} | val AUC {v['auc']:.2f} F1 {v['f1']:.2f} | {time.time()-t0:.0f}s")
        if v["auc"] > S["best"]: S["best"], S["best_g"] = float(v["auc"]), {k_: t_.clone() for k_, t_ in acc.items()}
        if r % SAVE_EVERY == 0 or r == ROUNDS: torch.save(S, CK)
    set_state(mod, S["best_g"]); torch.save(S["best_g"], BEST)
    P = {s: probs(mod, enc[s])[0] for s in EVS}; np.save(f"{FED}/ff_bal_{kind}_probs.npy", P, allow_pickle=True)
    print(f"[{kind}] xong | vòng tốt nhất {int(np.argmax([x['val_auc'] for x in S['LOG']]))+1} | val AUC {S['best']:.2f}")
    del mod; torch.cuda.empty_cache(); return P

Tương quan nguồn–nhãn trong tập train (tỉ lệ nhãn 1 = real):
col_0     0     1     n
row_0                  
0      0.92  0.08   313
1      0.99  0.01   126
2      0.88  0.12   976
3      0.25  0.75  1420
4      0.30  0.70  3103
5      0.39  0.61   427
6      0.66  0.34   459
client 0:
  n=3436 | giữ [np.str_('0'), np.str_('2'), np.str_('3'), np.str_('4'), np.str_('5'), np.str_('6')] (99% dữ liệu gốc) | bỏ (fake/real) ['1(36/1)']
client 1:
  n=601 | giữ [np.str_('2'), np.str_('3'), np.str_('4'), np.str_('5')] (89% dữ liệu gốc) | bỏ (fake/real) ['0(35/0)', '1(4/0)', '6(25/3)']
client 2:
  n=425 | giữ [np.str_('4')] (55% dữ liệu gốc) | bỏ (fake/real) ['0(19/1)', '1(77/0)', '2(51/4)', '3(4/13)', '5(1/5)', '6(17/0)']
client 3:
  n=2362 | giữ [np.str_('2'), np.str_('3'), np.str_('4'), np.str_('5'), np.str_('6')] (97% dữ liệu gốc) | bỏ (fake/real) ['0(52/2)', '1(8/0)']
Tham số CLIPModalMFND: ['mode']


In [ ]:
# Ô B1 — FedAvg chỉ ảnh, cân bằng theo nguồn
PIb = fed_bal("image")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[image] round  1 | loss 0.6939 | val AUC 49.73 F1 13.57 | 96s
[image] round  2 | loss 0.6938 | val AUC 51.03 F1 14.36 | 50s
[image] round  3 | loss 0.6933 | val AUC 52.72 F1 17.18 | 44s
[image] round  4 | loss 0.6929 | val AUC 54.50 F1 21.42 | 44s
[image] round  5 | loss 0.6926 | val AUC 56.07 F1 27.25 | 44s
[image] round  6 | loss 0.6922 | val AUC 57.39 F1 34.63 | 43s
[image] round  7 | loss 0.6911 | val AUC 58.51 F1 41.72 | 44s
[image] round  8 | loss 0.6909 | val AUC 59.23 F1 48.46 | 44s
[image] round  9 | loss 0.6901 | val AUC 59.84 F1 54.99 | 44s
[image] round 10 | loss 0.6893 | val AUC 60.39 F1 60.38 | 44s
[image] round 11 | loss 0.6880 | val AUC 60.70 F1 63.66 | 44s
[image] round 12 | loss 0.6868 | val AUC 61.08 F1 65.77 | 43s
[image] round 13 | loss 0.6861 | val AUC 61.48 F1 67.16 | 43s
[image] round 14 | loss 0.6839 | val AUC 61.66 F1 68.39 | 50s
[image] round 15 | loss 0.6837 | val AUC 61.76 F1 69.00 | 44s
[image] round 16 | loss 0.6798 | val AUC 61.95 F1 69.24 | 44s
[image] 

In [ ]:
# Ô B2 — FedAvg chỉ chữ, cân bằng theo nguồn
PTb = fed_bal("text")

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[text] round  1 | loss 0.6940 | val AUC 54.95 F1 6.60 | 44s
[text] round  2 | loss 0.6932 | val AUC 60.27 F1 22.39 | 44s
[text] round  3 | loss 0.6917 | val AUC 62.65 F1 25.82 | 44s
[text] round  4 | loss 0.6900 | val AUC 64.32 F1 26.09 | 44s
[text] round  5 | loss 0.6878 | val AUC 65.47 F1 25.85 | 43s
[text] round  6 | loss 0.6862 | val AUC 66.01 F1 25.73 | 44s
[text] round  7 | loss 0.6834 | val AUC 66.72 F1 25.85 | 44s
[text] round  8 | loss 0.6806 | val AUC 67.31 F1 26.69 | 44s
[text] round  9 | loss 0.6786 | val AUC 67.45 F1 27.11 | 44s
[text] round 10 | loss 0.6761 | val AUC 67.60 F1 29.21 | 44s
[text] round 11 | loss 0.6743 | val AUC 67.50 F1 31.59 | 44s
[text] round 12 | loss 0.6718 | val AUC 67.60 F1 33.71 | 43s
[text] round 13 | loss 0.6713 | val AUC 67.37 F1 36.72 | 43s
[text] round 14 | loss 0.6686 | val AUC 67.43 F1 39.19 | 43s
[text] round 15 | loss 0.6670 | val AUC 67.35 F1 42.59 | 43s
[text] round 16 | loss 0.6656 | val AUC 67.92 F1 44.70 | 44s
[text] round 17 | loss 0.

In [ ]:
# Ô B3 — So sánh: huấn luyện thường vs cân bằng theo nguồn (cùng giao thức FedSVLM-LF)
def _load_P(kind):
    mod, keys = build_modal(kind); set_state(mod, torch.load(f"{FED}/ff_{kind}_s0_best.pt"))
    P = {s: probs(mod, enc[s])[0] for s in EVS}; del mod; torch.cuda.empty_cache(); return P
PI0 = PI if "PI" in globals() else _load_P("image"); PT0 = PT if "PT" in globals() else _load_P("text")
if "PIb" not in globals(): PIb = np.load(f"{FED}/ff_bal_image_probs.npy", allow_pickle=True).item()
if "PTb" not in globals(): PTb = np.load(f"{FED}/ff_bal_text_probs.npy", allow_pickle=True).item()
avg2 = lambda A, B: {s: (A[s] + B[s]) / 2 for s in EVS}
CMP = {"Chỉ ảnh (thường)": PI0, "Chỉ ảnh (cân bằng nguồn)": PIb,
       "Chỉ chữ (thường)": PT0, "Chỉ chữ (cân bằng nguồn)": PTb,
       "FedSVLM-LF (thường, mốc)": avg2(PI0, PT0), "FedSVLM-LF (cân bằng nguồn)": avg2(PIb, PTb)}
rows, out = [], {}
for name, P in CMP.items():
    g, w_, per = within(P)
    row = {"Mô hình": name, "val AUC": round(roc_auc_score(enc["validation"].labels, P["validation"]) * 100, 1),
           "test AUC": round(roc_auc_score(enc["test"].labels, P["test"]) * 100, 1),
           "OOD AUC": round(float(np.mean([roc_auc_score(enc[s].labels, P[s]) * 100 for s in TESTS[1:]])), 1),
           "AUC gộp": round(float(g), 1), "TB trong từng nguồn": round(float(w_), 1)}
    rows.append(row); out[name] = {**row, "theo nguồn": per}
print(pd.DataFrame(rows).to_string(index=False))
for name in ("FedSVLM-LF (thường, mốc)", "FedSVLM-LF (cân bằng nguồn)"): print(name, "→", out[name]["theo nguồn"])
save_result("FineFake | Huấn luyện cân bằng theo nguồn (FedSVLM-LF)", out)

                    Mô hình  val AUC  test AUC  OOD AUC  AUC gộp  TB trong từng nguồn
           Chỉ ảnh (thường)     74.9      77.1     74.9     76.7                 57.3
   Chỉ ảnh (cân bằng nguồn)     62.5      64.5     64.6     65.8                 60.6
           Chỉ chữ (thường)     81.0      80.9     78.6     81.4                 58.8
   Chỉ chữ (cân bằng nguồn)     68.5      68.3     70.4     73.0                 62.1
   FedSVLM-LF (thường, mốc)     82.0      82.3     79.8     82.5                 59.1
FedSVLM-LF (cân bằng nguồn)     69.0      70.5     72.1     74.6                 63.9
FedSVLM-LF (thường, mốc) → {'apnews': np.float64(51.88077054599339), 'cnn': np.float64(43.551094998959975), 'reddit': np.float64(94.60295096082345), 'snope': np.float64(58.25252929200598), 'twitter': np.float64(54.70294082628884), 'washingtonpost': np.float64(51.57604832977967)}
FedSVLM-LF (cân bằng nguồn) → {'apnews': np.float64(52.23412743645276), 'cnn': np.float64(52.52577779098445), 'reddit'

{'FineFake | Tập trung Light-MFND (seed=0)': {'val': {'acc': 64.41124780316343,
   'f1': 75.13812154696133,
   'auc': 74.00415576540166},
  'test': {'acc': 64.94076349275998,
   'f1': 75.54331190694828,
   'auc': 76.25330421880163},
  'test_dedup': {'acc': 62.53588516746411,
   'f1': 73.23076923076923,
   'auc': 74.45682663535054},
  'ood_entertainment': {'acc': 62.55744795890781,
   'f1': 75.76552930883639,
   'auc': 78.42701471663194},
  'ood_entertainment_dedup': {'acc': 62.51698831204131,
   'f1': 75.74318381706244,
   'auc': 78.31498775096786},
  'ood_conflict': {'acc': 59.72060535506402,
   'f1': 70.62818336162988,
   'auc': 65.75605538207998},
  'ood_conflict_dedup': {'acc': 59.94117647058823,
   'f1': 70.9594882729211,
   'auc': 65.53707935462172},
  'OOD': {'acc': 61.13902665698592,
   'f1': 73.19685633523314,
   'auc': 72.09153504935597},
  'time_s': 253.6988034248352,
  'epochs_run': 4},
 'FineFake | Tập trung Light-MFND (20 epoch, chọn AUC, seed=0)': {'val': {'acc': 72.1880

In [ ]:
# Ô B4a — Cân bằng theo nguồn, SEED 1: định nghĩa hàm có seed + chạy nhánh ảnh
def build_modal(kind, seed=0):                      # seed=0 giữ nguyên hành vi cũ của B0/B3
    torch.manual_seed(seed); mod = (CLIPModalMFND(**{_kw: kind}) if _kw else CLIPModalMFND(kind)).to(device)
    keys = [n for n, p in mod.named_parameters() if p.requires_grad]
    set_state(mod, {k: v for k, v in INIT.items() if k in keys}); return mod, keys

def fed_bal_s(kind, seed, ROUNDS=20, SAVE_EVERY=5):
    mod, keys = build_modal(kind, seed); CK = f"{FED}/ck_ff_bal_{kind}_s{seed}.pt"; BEST = f"{FED}/ff_bal_{kind}_s{seed}_best.pt"
    S = torch.load(CK, weights_only=False) if os.path.exists(CK) else dict(g=get_state(mod, keys), best=-1.0, best_g=None, LOG=[], r=0)
    if S["r"]: print(f"Tiếp tục từ checkpoint, vòng {S['r']+1}")
    N = sum(len(p) for p in PARTS)
    for r in range(S["r"] + 1, ROUNDS + 1):
        t0 = time.time(); torch.manual_seed(seed * 1000 + r); g = S["g"]; acc = {k: torch.zeros_like(v) for k, v in g.items()}; L = []
        for k, part in enumerate(PARTS):
            set_state(mod, g)
            L.append(train_epochs(mod, enc["train"], bal_idx(part, np.random.default_rng(seed * 100000 + 1000 * r + k)), epochs=1))
            st = get_state(mod, keys); w = len(part) / N
            for key in acc: acc[key] += st[key] * w
        set_state(mod, acc); v = evaluate(mod, enc["validation"]); S["g"], S["r"] = acc, r
        S["LOG"].append(dict(round=r, val_auc=float(v["auc"]), val_f1=float(v["f1"]), loss=float(np.mean(L)), sec=time.time() - t0))
        print(f"[{kind} s{seed}] round {r:2d} | loss {np.mean(L):.4f} | val AUC {v['auc']:.2f} F1 {v['f1']:.2f} | {time.time()-t0:.0f}s")
        if v["auc"] > S["best"]: S["best"], S["best_g"] = float(v["auc"]), {k_: t_.clone() for k_, t_ in acc.items()}
        if r % SAVE_EVERY == 0 or r == ROUNDS: torch.save(S, CK)
    set_state(mod, S["best_g"]); torch.save(S["best_g"], BEST)
    P = {s: probs(mod, enc[s])[0] for s in EVS}; np.save(f"{FED}/ff_bal_{kind}_s{seed}_probs.npy", P, allow_pickle=True)
    print(f"[{kind} s{seed}] xong | vòng tốt nhất {int(np.argmax([x['val_auc'] for x in S['LOG']]))+1} | val AUC {S['best']:.2f}")
    del mod; torch.cuda.empty_cache(); return P

PIb1 = fed_bal_s("image", 1)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[image s1] round  1 | loss 0.6938 | val AUC 49.52 F1 13.32 | 74s
[image s1] round  2 | loss 0.6937 | val AUC 50.85 F1 14.76 | 44s
[image s1] round  3 | loss 0.6933 | val AUC 52.26 F1 17.42 | 44s
[image s1] round  4 | loss 0.6927 | val AUC 53.99 F1 21.90 | 44s
[image s1] round  5 | loss 0.6922 | val AUC 55.52 F1 28.09 | 44s
[image s1] round  6 | loss 0.6922 | val AUC 57.00 F1 34.95 | 44s
[image s1] round  7 | loss 0.6913 | val AUC 58.35 F1 43.29 | 44s
[image s1] round  8 | loss 0.6907 | val AUC 59.17 F1 49.67 | 44s
[image s1] round  9 | loss 0.6898 | val AUC 59.87 F1 55.69 | 44s
[image s1] round 10 | loss 0.6888 | val AUC 60.44 F1 61.56 | 44s
[image s1] round 11 | loss 0.6878 | val AUC 60.86 F1 64.31 | 44s
[image s1] round 12 | loss 0.6872 | val AUC 61.22 F1 66.14 | 44s
[image s1] round 13 | loss 0.6859 | val AUC 61.49 F1 67.57 | 44s
[image s1] round 14 | loss 0.6846 | val AUC 61.52 F1 68.57 | 44s
[image s1] round 15 | loss 0.6830 | val AUC 61.57 F1 69.14 | 44s
[image s1] round 16 | los

In [ ]:
# Ô B4b — Cân bằng theo nguồn, SEED 1, nhánh chữ
PTb1 = fed_bal_s("text", 1)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

[text s1] round  1 | loss 0.6941 | val AUC 55.26 F1 6.60 | 44s
[text s1] round  2 | loss 0.6929 | val AUC 59.89 F1 22.34 | 45s
[text s1] round  3 | loss 0.6913 | val AUC 61.89 F1 26.10 | 46s
[text s1] round  4 | loss 0.6897 | val AUC 63.46 F1 26.03 | 46s
[text s1] round  5 | loss 0.6880 | val AUC 64.95 F1 25.83 | 46s
[text s1] round  6 | loss 0.6866 | val AUC 65.25 F1 25.61 | 45s
[text s1] round  7 | loss 0.6834 | val AUC 65.78 F1 25.97 | 45s
[text s1] round  8 | loss 0.6811 | val AUC 66.38 F1 26.68 | 46s



----- [session interrupted by GPU quota after round 8; resumed below from the round-5 checkpoint] -----



Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Tiếp tục từ checkpoint, vòng 6
[text s1] round  6 | loss 0.6866 | val AUC 65.25 F1 25.61 | 59s
[text s1] round  7 | loss 0.6834 | val AUC 65.78 F1 25.97 | 41s
[text s1] round  8 | loss 0.6811 | val AUC 66.38 F1 26.68 | 42s
[text s1] round  9 | loss 0.6785 | val AUC 66.46 F1 27.19 | 41s
[text s1] round 10 | loss 0.6776 | val AUC 66.60 F1 28.80 | 42s
[text s1] round 11 | loss 0.6750 | val AUC 66.42 F1 30.92 | 41s
[text s1] round 12 | loss 0.6720 | val AUC 66.28 F1 33.71 | 41s
[text s1] round 13 | loss 0.6702 | val AUC 66.58 F1 36.26 | 42s
[text s1] round 14 | loss 0.6693 | val AUC 66.88 F1 40.02 | 42s
[text s1] round 15 | loss 0.6682 | val AUC 66.87 F1 43.29 | 41s
[text s1] round 16 | loss 0.6661 | val AUC 66.58 F1 45.86 | 41s
[text s1] round 17 | loss 0.6634 | val AUC 66.93 F1 48.07 | 41s
[text s1] round 18 | loss 0.6616 | val AUC 67.58 F1 49.51 | 41s
[text s1] round 19 | loss 0.6620 | val AUC 68.05 F1 50.98 | 41s
[text s1] round 20 | loss 0.6577 | val AUC 68.96 F1 52.67 | 41s
[text s1]

In [ ]:
# Ô B5a — Huấn luyện THƯỜNG (đối chứng), SEED 1, nhánh ảnh
ff_fed("image", SEED_RUN=1)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

round  1 | loss 0.6942 | val Acc 45.39 F1 20.58 AUC 51.48 | 41s
round  2 | loss 0.6930 | val Acc 48.24 F1 34.48 AUC 55.54 | 40s
round  3 | loss 0.6915 | val Acc 55.05 F1 54.10 AUC 59.97 | 41s
round  4 | loss 0.6897 | val Acc 61.12 F1 67.38 AUC 63.65 | 41s
round  5 | loss 0.6874 | val Acc 63.58 F1 73.13 AUC 66.49 | 41s
round  6 | loss 0.6843 | val Acc 63.01 F1 74.14 AUC 68.37 | 41s
round  7 | loss 0.6801 | val Acc 63.14 F1 74.51 AUC 69.49 | 41s
round  8 | loss 0.6744 | val Acc 63.14 F1 74.57 AUC 70.09 | 41s
round  9 | loss 0.6681 | val Acc 63.14 F1 74.57 AUC 70.61 | 41s
round 10 | loss 0.6629 | val Acc 63.14 F1 74.55 AUC 71.32 | 41s
round 11 | loss 0.6585 | val Acc 63.18 F1 74.58 AUC 72.09 | 41s
round 12 | loss 0.6538 | val Acc 63.27 F1 74.62 AUC 72.77 | 41s
round 13 | loss 0.6492 | val Acc 64.24 F1 75.09 AUC 73.29 | 40s
round 14 | loss 0.6448 | val Acc 64.41 F1 75.17 AUC 73.68 | 40s
round 15 | loss 0.6390 | val Acc 64.41 F1 75.17 AUC 73.99 | 40s
round 16 | loss 0.6338 | val Acc 64.67 F

'/content/drive/MyDrive/NLP_project/fed/finefake/ff_image_s1_best.pt'

In [ ]:
# Ô B5b — Huấn luyện THƯỜNG (đối chứng), SEED 1, nhánh chữ
ff_fed("text", SEED_RUN=1)

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

round  1 | loss 0.6925 | val Acc 46.70 F1 12.92 AUC 62.26 | 40s
round  2 | loss 0.6888 | val Acc 52.33 F1 30.31 AUC 68.67 | 40s
round  3 | loss 0.6841 | val Acc 53.43 F1 34.16 AUC 72.20 | 40s
round  4 | loss 0.6783 | val Acc 55.54 F1 40.47 AUC 74.60 | 41s
round  5 | loss 0.6695 | val Acc 58.00 F1 47.70 AUC 76.15 | 41s
round  6 | loss 0.6523 | val Acc 63.97 F1 60.92 AUC 77.49 | 41s
round  7 | loss 0.6307 | val Acc 70.65 F1 72.44 AUC 78.27 | 41s
round  8 | loss 0.6156 | val Acc 74.56 F1 78.19 AUC 78.51 | 41s
round  9 | loss 0.6038 | val Acc 75.88 F1 80.03 AUC 78.65 | 41s
round 10 | loss 0.5927 | val Acc 75.83 F1 80.29 AUC 78.77 | 40s
round 11 | loss 0.5820 | val Acc 75.62 F1 80.28 AUC 78.87 | 40s
round 12 | loss 0.5717 | val Acc 75.70 F1 80.45 AUC 79.03 | 40s
round 13 | loss 0.5621 | val Acc 75.62 F1 80.40 AUC 79.25 | 41s
round 14 | loss 0.5540 | val Acc 75.48 F1 80.31 AUC 79.51 | 40s
round 15 | loss 0.5469 | val Acc 75.48 F1 80.31 AUC 79.80 | 40s
round 16 | loss 0.5402 | val Acc 75.57 F

'/content/drive/MyDrive/NLP_project/fed/finefake/ff_text_s1_best.pt'

In [ ]:
# Ô B6-pre — tạo PI0, PT0 (FedSVLM-LF thường, seed 0) từ checkpoint đã lưu
PI0 = _P_ckpt("image", 0); PT0 = _P_ckpt("text", 0)
print("OK:", {s: len(PI0[s]) for s in EVS})

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

OK: {'validation': 2276, 'test': 2279, 'ood_entertainment': 3699, 'ood_conflict': 1718}


In [ ]:
# Ô B6-within — định nghĩa lại within() (thay cho K2) + tự kiểm tra với số cũ
def _src(s):
    base = NPY if os.path.exists(f"{NPY}/{s}_plat.npy") else f"{FED}/npy"
    codes = np.load(f"{base}/{s}_plat.npy", allow_pickle=True).astype(str)
    mp = f"{base}/{s}_meta.csv"
    if os.path.exists(mp):
        meta = pd.read_csv(mp)
        for c in meta.columns:
            v = meta[c].astype(str)
            if len(v) == len(codes) and v.nunique() <= 20 and v.str.lower().str.contains("reddit").any(): return v.values
    return codes
def within(P, sets=None):
    sets = sets or [s for s in EVS if s != "validation"]
    p = np.concatenate([P[s] for s in sets]); y = np.concatenate([np.asarray(enc[s].labels) for s in sets])
    src = np.concatenate([_src(s) for s in sets]); per = {}
    for u in np.unique(src):
        mk = src == u
        if len(np.unique(y[mk])) == 2: per[u] = roc_auc_score(y[mk], p[mk]) * 100
    return roc_auc_score(y, p) * 100, float(np.mean(list(per.values()))), per

g, w_, per = within({s: (PI0[s] + PT0[s]) / 2 for s in EVS})
print(f"Kiểm tra: AUC gộp {g:.1f} (cũ 82.5) | TB trong từng nguồn {w_:.1f} (cũ 59.1)")
print({k: round(v, 1) for k, v in per.items()})
print("số mẫu theo nguồn:", pd.Series(np.concatenate([_src(s) for s in EVS if s != 'validation'])).value_counts().to_dict())

Kiểm tra: AUC gộp 82.5 (cũ 82.5) | TB trong từng nguồn 59.1 (cũ 59.1)
{'apnews': np.float64(51.9), 'cnn': np.float64(43.6), 'reddit': np.float64(94.6), 'snope': np.float64(58.3), 'twitter': np.float64(54.7), 'washingtonpost': np.float64(51.6)}
số mẫu theo nguồn: {'snope': 3420, 'reddit': 2129, 'cnn': 995, 'twitter': 375, 'washingtonpost': 373, 'apnews': 314, 'cdc_gov': 90}


In [ ]:
# Ô B6 — Tổng hợp 2 seed: thường vs cân bằng theo nguồn (FedSVLM-LF)
import glob
def _P_ckpt(kind, seed):
    path = f"{FED}/ff_{kind}_s{seed}_best.pt"
    if not os.path.exists(path):
        print("Không thấy", path, "| các file có:", sorted(os.path.basename(x) for x in glob.glob(f"{FED}/ff_{kind}*best.pt"))); raise FileNotFoundError(path)
    mod, keys = build_modal(kind, seed); set_state(mod, torch.load(path))
    P = {s: probs(mod, enc[s])[0] for s in EVS}; del mod; torch.cuda.empty_cache(); return P
_npy = lambda kind, seed: np.load(f"{FED}/ff_bal_{kind}_s{seed}_probs.npy" if seed else f"{FED}/ff_bal_{kind}_probs.npy", allow_pickle=True).item()
avg2 = lambda A, B: {s: (A[s] + B[s]) / 2 for s in EVS}
RUNS = {("thường", 0): avg2(PI0, PT0), ("thường", 1): avg2(_P_ckpt("image", 1), _P_ckpt("text", 1)),
        ("cân bằng", 0): avg2(_npy("image", 0), _npy("text", 0)), ("cân bằng", 1): avg2(_npy("image", 1), _npy("text", 1))}
def _row(P):
    g, w_, per = within(P)
    r = {"val AUC": roc_auc_score(enc["validation"].labels, P["validation"]) * 100,
         "test AUC": roc_auc_score(enc["test"].labels, P["test"]) * 100,
         "OOD AUC": float(np.mean([roc_auc_score(enc[s].labels, P[s]) * 100 for s in TESTS[1:]])),
         "AUC gộp": float(g), "TB trong từng nguồn": float(w_)}
    r.update({k: float(v) for k, v in per.items()}); return r
R = {k: _row(P) for k, P in RUNS.items()}
df = pd.DataFrame([{"Cách huấn luyện": a, "seed": s, **{k: round(v, 1) for k, v in R[(a, s)].items()}} for a, s in RUNS])
print(df.to_string(index=False)); print()
cols = [c for c in df.columns if c not in ("Cách huấn luyện", "seed")]
summ = df.groupby("Cách huấn luyện")[cols].agg(["mean", "std"]).round(1)
print("Trung bình ± độ lệch chuẩn qua 2 seed:"); print(summ.T.to_string())
print("\nChênh lệch (cân bằng − thường) theo từng seed, cột 'TB trong từng nguồn':",
      [round(R[("cân bằng", s)]["TB trong từng nguồn"] - R[("thường", s)]["TB trong từng nguồn"], 1) for s in (0, 1)])
save_result("FineFake | Cân bằng theo nguồn: 2 seed (thường vs cân bằng)",
            {f"{a} | seed {s}": {k: round(v, 2) for k, v in R[(a, s)].items()} for a, s in RUNS})

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Cách huấn luyện  seed  val AUC  test AUC  OOD AUC  AUC gộp  TB trong từng nguồn  apnews  cnn  reddit  snope  twitter  washingtonpost
         thường     0     82.0      82.3     79.8     82.5                 59.1    51.9 43.6    94.6   58.3     54.7            51.6
         thường     1     82.0      82.4     79.8     82.4                 59.0    51.1 44.1    94.6   58.3     54.4            51.5
       cân bằng     0     69.0      70.5     72.1     74.6                 63.9    52.2 52.5    96.8   62.9     65.2            53.5
       cân bằng     1     69.1      70.8     72.5     74.9                 64.3    52.5 53.9    96.8   63.4     66.7            52.8

Trung bình ± độ lệch chuẩn qua 2 seed:
Cách huấn luyện           cân bằng  thường
val AUC             mean      69.0    82.0
                    std        0.1     0.0
test AUC            mean      70.6    82.4
                    std        0.2     0.1
OOD AUC             mean      72.3    79.8
                    std        0.3   

{'FineFake | Tập trung Light-MFND (seed=0)': {'val': {'acc': 64.41124780316343,
   'f1': 75.13812154696133,
   'auc': 74.00415576540166},
  'test': {'acc': 64.94076349275998,
   'f1': 75.54331190694828,
   'auc': 76.25330421880163},
  'test_dedup': {'acc': 62.53588516746411,
   'f1': 73.23076923076923,
   'auc': 74.45682663535054},
  'ood_entertainment': {'acc': 62.55744795890781,
   'f1': 75.76552930883639,
   'auc': 78.42701471663194},
  'ood_entertainment_dedup': {'acc': 62.51698831204131,
   'f1': 75.74318381706244,
   'auc': 78.31498775096786},
  'ood_conflict': {'acc': 59.72060535506402,
   'f1': 70.62818336162988,
   'auc': 65.75605538207998},
  'ood_conflict_dedup': {'acc': 59.94117647058823,
   'f1': 70.9594882729211,
   'auc': 65.53707935462172},
  'OOD': {'acc': 61.13902665698592,
   'f1': 73.19685633523314,
   'auc': 72.09153504935597},
  'time_s': 253.6988034248352,
  'epochs_run': 4},
 'FineFake | Tập trung Light-MFND (20 epoch, chọn AUC, seed=0)': {'val': {'acc': 72.1880

## Finish
Flush all files to Google Drive.

In [ ]:
# Ô cuối — Đẩy hết dữ liệu lên Drive và ngắt kết nối Drive an toàn
from google.colab import drive
drive.flush_and_unmount()
print("Đã lưu xong lên Drive và ngắt kết nối.")

Đã lưu xong lên Drive và ngắt kết nối.
